In [1]:
# STEP 1 (fixed): Build EfficientNet-B0 and run a forward sanity on the SAME device
import torch
from torch import nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_dim = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(in_dim, 2)

# <<< important line: put the model on the same device as the tensor >>>
effnet = effnet.to(device)

# forward sanity on the same device
x = torch.randn(2, 3, 224, 224, device=device)
with torch.no_grad():
    y = effnet(x)
total = sum(p.numel() for p in effnet.parameters())
trainable = sum(p.numel() for p in effnet.parameters() if p.requires_grad)
print(f"EfficientNet-B0 params (M): total={total/1e6:.2f}, trainable={trainable/1e6:.2f}")
print("Forward OK, logits shape:", tuple(y.shape))


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 52.7MB/s]


EfficientNet-B0 params (M): total=4.01, trainable=4.01
Forward OK, logits shape: (2, 2)


In [3]:
# Rebuild DataLoaders from your existing splits (num_workers=0)
import os, random, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T

# Use the ViT-Model splits you created earlier
SPLITS_DIR = "///workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet_Model/metadata/splits"
TRAIN_CSV, VAL_CSV, TEST_CSV = [f"{SPLITS_DIR}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]), "Split CSVs not found."

LABEL2ID = {"healthy":0, "cancer":1}

# --- light spec augment (minority gets slightly stronger) ---
class SpecAug2D(torch.nn.Module):
    def __init__(self, T_mask=30, F_mask=20, num_T=2, num_F=2, p=0.7):
        super().__init__(); self.T_mask=T_mask; self.F_mask=F_mask; self.num_T=num_T; self.num_F=num_F; self.p=p
    def forward(self, x):
        if not self.training or torch.rand(())>self.p: return x
        c,H,W = x.shape
        for _ in range(self.num_T):
            w = int(torch.randint(0, self.T_mask+1, (1,)))
            if 0<w<W:
                t0 = int(torch.randint(0, W-w+1, (1,))); x[:, :, t0:t0+w] = 0.0
        for _ in range(self.num_F):
            h = int(torch.randint(0, self.F_mask+1, (1,)))
            if 0<h<H:
                f0 = int(torch.randint(0, H-h+1, (1,))); x[:, f0:f0+h, :] = 0.0
        return x

class RandTimeShift(torch.nn.Module):
    def __init__(self, max_frac=0.05, p=0.5):
        super().__init__(); self.max_frac=max_frac; self.p=p
    def forward(self, x):
        if not self.training or torch.rand(())>self.p: return x
        _,H,W = x.shape
        max_pix = max(1, int(self.max_frac*W))
        s = int(torch.randint(-max_pix, max_pix+1, (1,)))
        if s==0: return x
        pad = torch.zeros((1,H,abs(s)), dtype=x.dtype, device=x.device)
        return torch.cat([pad, x[:,:,:-s]], dim=2) if s>0 else torch.cat([x[:,:, -s:], pad], dim=2)

class AddNoise(torch.nn.Module):
    def __init__(self, lo=0.005, hi=0.02, p=0.3):
        super().__init__(); self.lo=lo; self.hi=hi; self.p=p
    def forward(self, x):
        if not self.training or torch.rand(())>self.p: return x
        sigma = float(self.lo + (self.hi-self.lo)*torch.rand(()))
        return (x + torch.randn_like(x)*sigma).clamp(0.0, 1.0)

def train_tf(minority=False):
    spec = SpecAug2D(T_mask=40 if minority else 30, F_mask=24 if minority else 20, p=0.9 if minority else 0.7)
    shift= RandTimeShift(max_frac=0.08 if minority else 0.05, p=0.7 if minority else 0.5)
    noise= AddNoise(lo=0.005, hi=0.02, p=0.6 if minority else 0.3)
    return T.Compose([
        T.Lambda(lambda p: p.convert("L")),
        T.ToTensor(),
        spec, shift, noise,
        T.Lambda(lambda t: t.repeat(3,1,1)),  # 3-ch
        T.RandomErasing(p=0.10, scale=(0.01,0.05), ratio=(0.3,3.3), value=0.0),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class CSVImageDS(Dataset):
    def __init__(self, csv_path, tf_h, tf_c):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df; self.tf_h=tf_h; self.tf_c=tf_c
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = LABEL2ID[str(r["label"]).lower().strip()]
        x = Image.open(r["filepath"])
        tf = self.tf_c if y==1 else self.tf_h
        return tf(x), torch.tensor(y, dtype=torch.long)

train_ds = CSVImageDS(TRAIN_CSV, tf_h=train_tf(False), tf_c=train_tf(True))
val_ds   = CSVImageDS(VAL_CSV,   tf_h=eval_tf,         tf_c=eval_tf)
test_ds  = CSVImageDS(TEST_CSV,  tf_h=eval_tf,         tf_c=eval_tf)

# oversampling like before
y_train = train_ds.df["label"].map(LABEL2ID).values
counts = np.bincount(y_train, minlength=2).astype(np.float32)
class_sampling_w = (counts.sum() / (2.0*np.maximum(counts,1.0))).astype(np.float32)
weights = class_sampling_w[y_train]
sampler = WeightedRandomSampler(weights.tolist(), num_samples=len(weights), replacement=True)

# loaders (no multiprocessing to avoid SHM issues)
BATCH=32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False)

# sanity print
xb, yb = next(iter(train_loader))
print("num_workers:", train_loader.num_workers, val_loader.num_workers, test_loader.num_workers)
print("Train batch:", tuple(xb.shape), "| first labels:", yb[:12].tolist())
print("Train counts [healthy,cancer]:", counts.astype(int).tolist())
print("Sampler class weights:", class_sampling_w.tolist())
print("Sizes -> train/val/test:", len(train_ds), len(val_ds), len(test_ds))


num_workers: 0 0 0
Train batch: (32, 3, 224, 224) | first labels: [0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0]
Train counts [healthy,cancer]: [2569, 21]
Sampler class weights: [0.5040872097015381, 61.66666793823242]
Sizes -> train/val/test: 2590 740 370


In [4]:
# Train EfficientNet-B0 with SAME loaders/sampler/class-weights
import os, numpy as np, torch
from torch import nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

assert 'train_loader' in globals() and 'val_loader' in globals() and 'test_loader' in globals()
assert 'train_ds' in globals() and 'effnet' in globals()
device = next(effnet.parameters()).device
model  = effnet  # already on device

# class-weighted CE from TRAIN
LABEL2ID = {"healthy":0, "cancer":1}
y_train = train_ds.df["label"].map(LABEL2ID).values
counts  = np.bincount(y_train, minlength=2)
class_w = torch.tensor([counts.sum()/(2*max(1,counts[0])),
                        counts.sum()/(2*max(1,counts[1]))],
                       dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_w)

opt   = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train)
    losses, ys, ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        probs = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
        ys.extend(yb.detach().cpu().numpy().tolist())
        ps.extend(probs.tolist())
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps >= 0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(ys, ps)
    except ValueError: auc = float('nan')
    return float(np.mean(losses)), acc, prec, rec, f1, auc

EPOCHS, PATIENCE = 15, 4
best_auc, best_epoch, best_state, no_improve = -1.0, -1, None, 0

CKPT_DIR = "//workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
SAVE_PATH = os.path.join(CKPT_DIR, "efficientnet_b0_best.pt")

for ep in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | "
          f"val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1] > best_auc:
        best_auc, best_epoch = vl[-1], ep
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        try:
            torch.save(best_state, SAVE_PATH)
        except OSError:
            SAVE_PATH = os.path.join(CKPT_DIR, "efficientnet_b0_best_fp16.pt")
            best_state = {k: (v.half() if v.dtype.is_floating_point else v) for k,v in best_state.items()}
            torch.save(best_state, SAVE_PATH)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {ep} (best val AUC at {best_epoch}: {best_auc:.3f})")
            break

print(f"\nBest epoch: {best_epoch} | Best val AUC: {best_auc:.3f}")
print("Saved:", SAVE_PATH)

# TEST with best weights
if best_state is not None:
    model.load_state_dict(best_state, strict=False)
test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = run_epoch(test_loader, train=False)
print("\nTEST METRICS")
print(f"loss {test_loss:.4f} | acc {test_acc:.3f} | prec {test_prec:.3f} | rec {test_rec:.3f} | f1 {test_f1:.3f} | auc {test_auc:.3f}")


Epoch 01 | train loss 0.0855 f1 0.964 auc 0.995 | val loss 0.0906 f1 0.400 auc 1.000
Epoch 02 | train loss 0.0016 f1 0.992 auc 1.000 | val loss 0.0740 f1 0.400 auc 0.999
Epoch 03 | train loss 0.0018 f1 0.994 auc 1.000 | val loss 0.0524 f1 0.706 auc 1.000
Epoch 04 | train loss 0.0011 f1 0.995 auc 0.999 | val loss 0.0438 f1 0.706 auc 1.000
Epoch 05 | train loss 0.0007 f1 0.997 auc 1.000 | val loss 0.0311 f1 0.706 auc 1.000
Early stopping at epoch 5 (best val AUC at 1: 1.000)

Best epoch: 1 | Best val AUC: 1.000
Saved: //workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model/checkpoints/efficientnet_b0_best.pt

TEST METRICS
loss 0.0921 | acc 0.986 | prec 0.375 | rec 1.000 | f1 0.545 | auc 1.000


In [5]:
# EfficientNet-B0 — threshold tuning on VAL → apply to TEST + save predictions
import os, numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score

# paths
EXP_DIR = "//workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model"
CKPT    = f"{EXP_DIR}/checkpoints/efficientnet_b0_best.pt"
os.makedirs(f"{EXP_DIR}/predictions", exist_ok=True)
os.makedirs(f"{EXP_DIR}/reports", exist_ok=True)
OUTCSV  = f"{EXP_DIR}/predictions/efficientnet_b0_val_test_predictions.csv"
TAU_TXT = f"{EXP_DIR}/reports/operating_threshold.txt"

# model on the right device
device = next(effnet.parameters()).device if 'effnet' in globals() else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'effnet' not in globals():
    from torchvision import models
    from torch import nn
    effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_dim = effnet.classifier[1].in_features
    effnet.classifier[1] = nn.Linear(in_dim, 2)
    effnet = effnet.to(device)

state = torch.load(CKPT, map_location=device)
effnet.load_state_dict(state, strict=False)
effnet.eval()

def collect_preds(loader, dataset):
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pr = torch.softmax(effnet(xb), dim=1)[:,1].cpu().numpy()
            probs.extend(pr); ys.extend(yb.numpy())
    df = dataset.df.copy()
    df["y_true"] = ys
    df["prob_cancer"] = probs
    return df

# use your existing loaders/datasets
val_df  = collect_preds(val_loader,  val_ds)
test_df = collect_preds(test_loader, test_ds)

def metrics_at(y, p, tau):
    yhat = (p >= tau).astype(int)
    acc = accuracy_score(y, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    return acc, prec, rec, f1

def choose_tau_unique(y, p, prec_target=0.98):
    ps = np.unique(p)
    mids = [(ps[i] + ps[i+1]) / 2.0 for i in range(len(ps)-1)]
    cands = np.array([0.0] + mids + [1.0])
    # try precision target first (max recall)
    best = None
    for t in cands:
        acc,pr,re,f1 = metrics_at(y, p, t)
        if pr >= prec_target:
            item = (re, f1, t, acc, pr)
            if best is None or item > best:
                best = item
    if best is not None:
        re,f1,t,acc,pr = best
        return float(t), {"policy":"prec>=0.98","acc":acc,"prec":pr,"rec":re,"f1":f1}
    # fallback: best F1
    best = (-1, None, None, None)
    for t in cands:
        acc,pr,re,f1 = metrics_at(y,p,t)
        if f1 > best[0]:
            best = (f1, t, (acc,pr,re))
    f1,t,(acc,pr,re) = best
    return float(t), {"policy":"bestF1","acc":acc,"prec":pr,"rec":re,"f1":f1}

# choose τ* on VAL
yv, pv = val_df["y_true"].values,  val_df["prob_cancer"].values
tau_star, val_sum = choose_tau_unique(yv, pv, prec_target=0.98)
print(f"Chosen τ*={tau_star:.6f}  policy={val_sum['policy']} | VAL acc={val_sum['acc']:.3f} prec={val_sum['prec']:.3f} rec={val_sum['rec']:.3f} f1={val_sum['f1']:.3f}")

# apply to TEST
yt, pt = test_df["y_true"].values, test_df["prob_cancer"].values
acc,pr,re,f1 = metrics_at(yt, pt, tau_star)
auc = roc_auc_score(yt, pt)
cm  = confusion_matrix(yt, (pt>=tau_star).astype(int), labels=[0,1])
print(f"[TEST @ τ*] acc={acc:.3f} prec={pr:.3f} rec={re:.3f} f1={f1:.3f} auc={auc:.3f}")
print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)

# Kaggle-only sanity
def kg(df): return df[df["source"].str.contains("kaggle_", na=False)]
for name, d in [("VAL", kg(val_df)), ("TEST", kg(test_df))]:
    if len(d):
        y, p = d["y_true"].values, d["prob_cancer"].values
        a,pp,rr,ff = metrics_at(y, p, tau_star)
        cmk = confusion_matrix(y, (p>=tau_star).astype(int), labels=[0,1])
        print(f"[{name} Kaggle-only @ τ*] acc={a:.3f} prec={pp:.3f} rec={rr:.3f} f1={ff:.3f}")
        print("  CM:\n", cmk)

# save predictions + τ*
pd.concat([val_df.assign(split='val'), test_df.assign(split='test')]).to_csv(OUTCSV, index=False)
with open(TAU_TXT, "w") as f:
    f.write(f"{tau_star:.6f}\nchosen='{val_sum['policy']}' via unique-probability search on VAL\n")
print(f"\n✅ Saved predictions to: {OUTCSV}")
print(f"τ* written to: {TAU_TXT}")


Chosen τ*=0.998690  policy=prec>=0.98 | VAL acc=1.000 prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[367   0]
 [  0   3]]
[VAL Kaggle-only @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000
  CM:
 [[5 0]
 [0 6]]
[TEST Kaggle-only @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000
  CM:
 [[2 0]
 [0 3]]

✅ Saved predictions to: //workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model/predictions/efficientnet_b0_val_test_predictions.csv
τ* written to: //workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model/reports/operating_threshold.txt


In [6]:
# Fix the unpacking and write the manifest again
import os, json, pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

EXP = "//workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model"
CKPT = f"{EXP}/checkpoints/efficientnet_b0_best.pt"
PRED = f"{EXP}/predictions/efficientnet_b0_val_test_predictions.csv"
TAU  = f"{EXP}/reports/operating_threshold.txt"
assert all(os.path.exists(p) for p in [CKPT,PRED,TAU]), "Missing one of CKPT/PRED/TAU."

df = pd.read_csv(PRED)
tau_star = float(open(TAU).read().strip().splitlines()[0])

def at_tau(d, t):
    y = d["y_true"].values; p = d["prob_cancer"].values
    yhat = (p>=t).astype(int)
    acc = accuracy_score(y, yhat)
    prec, rec, f1, support = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    auc = roc_auc_score(y, p)
    cm  = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    return dict(acc=float(acc), prec=float(prec), rec=float(rec), f1=float(f1),
                auc=float(auc), cm=cm, tau=float(t))

summary = {
    "val":  at_tau(df[df.split=="val"],  tau_star),
    "test": at_tau(df[df.split=="test"], tau_star),
}

manifest = {
    "model": "EfficientNet-B0",
    "paths": {"checkpoint":CKPT,"predictions_csv":PRED,"operating_threshold_txt":TAU},
    "tau_star": tau_star,
    "metrics": summary,
}
with open(f"{EXP}/manifest_effnet.json","w") as f: json.dump(manifest, f, indent=2)

print("✅ EfficientNet archived")
print("VAL @τ* ->", summary["val"])
print("TEST @τ*->", summary["test"])
print("Manifest ->", f"{EXP}/manifest_effnet.json")


✅ EfficientNet archived
VAL @τ* -> {'acc': 1.0, 'prec': 1.0, 'rec': 1.0, 'f1': 1.0, 'auc': 1.0, 'cm': [[734, 0], [0, 6]], 'tau': 0.99869}
TEST @τ*-> {'acc': 1.0, 'prec': 1.0, 'rec': 1.0, 'f1': 1.0, 'auc': 1.0, 'cm': [[367, 0], [0, 3]], 'tau': 0.99869}
Manifest -> //workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet-Model/manifest_effnet.json


In [11]:
# 1) Auto-find base splits (train/val/test.csv) anywhere under the project
import os, pandas as pd

PROJECT_ROOT = "/workspaces/cmp9137-advanced-machine-learning"
need = {"train.csv","val.csv","test.csv"}

found_dirs = []
for root, _, files in os.walk(PROJECT_ROOT):
    if need.issubset(set(files)):
        found_dirs.append(root)

print("Found split directories:")
for i, d in enumerate(found_dirs):
    print(f"  [{i}] {d}")
assert len(found_dirs) > 0, "No directories containing train/val/test.csv were found."

# Prefer the ViT-Model/metadata/splits path if present
preferred_end = "/CMP9137 Advanced Machine Learning/ViT-Model/metadata/splits"
BASE_SPLITS = None
for d in found_dirs:
    if d.replace("\\","/").endswith(preferred_end):
        BASE_SPLITS = d
        break
if BASE_SPLITS is None:
    BASE_SPLITS = found_dirs[0]  # fallback: first found
print("\nUsing base splits:", BASE_SPLITS)

# 2) Load and build LOSO folds (hold out each healthy source)
train = pd.read_csv(os.path.join(BASE_SPLITS, "train.csv"))
val   = pd.read_csv(os.path.join(BASE_SPLITS, "val.csv"))
test  = pd.read_csv(os.path.join(BASE_SPLITS, "test.csv"))

def save_split(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

def counts(title, df):
    print(f"{title} n={len(df)}")
    print("  by label:\n", df["label"].value_counts(dropna=False))
    print("  by source:\n", df["source"].value_counts(dropna=False).head(10))

OUTROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
heldouts = ["Cough_Audio_Coswera", "Cough_Audio_COUGHVID"]

for h in heldouts:
    fold_dir = os.path.join(OUTROOT, h.replace(" ","_"))
    # Train/Val: drop the held-out healthy source entirely
    tr = train[train["source"] != h].reset_index(drop=True)
    vl = val[val["source"] != h].reset_index(drop=True)
    # Test: healthy from the held-out source + all cancer from base TEST (Kaggle)
    te_neg = test[(test["source"] == h) & (test["label"].str.lower()=="healthy")]
    te_pos = test[test["label"].str.lower()=="cancer"]
    te = pd.concat([te_neg, te_pos], axis=0).drop_duplicates(subset=["filepath"]).reset_index(drop=True)

    save_split(tr, os.path.join(fold_dir, "train.csv"))
    save_split(vl, os.path.join(fold_dir, "val.csv"))
    save_split(te, os.path.join(fold_dir, "test.csv"))

    print("\n" + "="*68)
    print(f"LOSO fold (held-out healthy source): {h}")
    counts("TRAIN", tr); counts("VAL", vl); counts("TEST", te)

print("\n✅ LOSO split CSVs written under:", OUTROOT)


Found split directories:
  [0] /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet_Model/metadata/splits
  [1] /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/metadata/splits
  [2] /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/ViT-Model/ViT_Model_data_code/metadata/splits

Using base splits: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/EffNet_Model/metadata/splits

LOSO fold (held-out healthy source): Cough_Audio_Coswera
TRAIN n=624
  by label:
 label
healthy    603
cancer      21
Name: count, dtype: int64
  by source:
 source
Cough_Audio_COUGHVID    586
kaggle_cancer_raw        21
kaggle_normal_raw        17
Name: count, dtype: int64
VAL n=178
  by label:
 label
healthy    172
cancer       6
Name: count, dtype: int64
  by source:
 source
Cough_Audio_COUGHVID    167
kaggle_cancer_raw         6
kaggle_n

In [12]:
# LOSO (hold-out Coswera) — rebuild loaders
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T

FOLD_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera"
TRAIN_CSV, VAL_CSV, TEST_CSV = [f"{FOLD_DIR}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]), "LOSO CSVs not found."

LABEL2ID = {"healthy":0, "cancer":1}

# augment (slightly stronger for minority class)
class SpecAug2D(torch.nn.Module):
    def __init__(self, T_mask=30, F_mask=20, num_T=2, num_F=2, p=0.7):
        super().__init__(); self.T_mask=T_mask; self.F_mask=F_mask; self.num_T=num_T; self.num_F=num_F; self.p=p
    def forward(self, x):
        if not self.training or torch.rand(())>self.p: return x
        c,H,W = x.shape
        for _ in range(self.num_T):
            w = int(torch.randint(0, self.T_mask+1, (1,)))
            if 0<w<W:
                t0 = int(torch.randint(0, W-w+1, (1,))); x[:, :, t0:t0+w] = 0.0
        for _ in range(self.num_F):
            h = int(torch.randint(0, self.F_mask+1, (1,)))
            if 0<h<H:
                f0 = int(torch.randint(0, H-h+1, (1,))); x[:, f0:f0+h, :] = 0.0
        return x

def train_tf(minority=False):
    spec = SpecAug2D(T_mask=40 if minority else 30, F_mask=24 if minority else 20, p=0.9 if minority else 0.7)
    return T.Compose([
        T.Lambda(lambda p: p.convert("L")),
        T.ToTensor(),
        spec,
        T.Lambda(lambda t: t.repeat(3,1,1)),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class CSVImageDS(Dataset):
    def __init__(self, csv_path, tf_h, tf_c):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df; self.tf_h=tf_h; self.tf_c=tf_c
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = LABEL2ID[str(r["label"]).lower().strip()]
        x = Image.open(r["filepath"])
        tf = self.tf_c if y==1 else self.tf_h
        return tf(x), torch.tensor(y, dtype=torch.long)

train_ds = CSVImageDS(TRAIN_CSV, tf_h=train_tf(False), tf_c=train_tf(True))
val_ds   = CSVImageDS(VAL_CSV,   tf_h=eval_tf,         tf_c=eval_tf)
test_ds  = CSVImageDS(TEST_CSV,  tf_h=eval_tf,         tf_c=eval_tf)

# oversampling on this fold’s TRAIN
y_train = train_ds.df["label"].map(LABEL2ID).values
counts  = np.bincount(y_train, minlength=2).astype(np.float32)
class_sampling_w = (counts.sum() / (2.0*np.maximum(counts,1.0))).astype(np.float32)
weights = class_sampling_w[y_train]
sampler = WeightedRandomSampler(weights.tolist(), num_samples=len(weights), replacement=True)

# loaders (no multiprocessing)
BATCH=32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False)

# sanity
xb, yb = next(iter(train_loader))
print("FOLD:", FOLD_DIR.split("/")[-1])
print("num_workers:", train_loader.num_workers, val_loader.num_workers, test_loader.num_workers)
print("Train batch:", tuple(xb.shape), "| first labels:", yb[:12].tolist())
print("Train counts [healthy,cancer]:", counts.astype(int).tolist())
print("Sampler class weights:", class_sampling_w.tolist())
print("Sizes -> train/val/test:", len(train_ds), len(val_ds), len(test_ds))


FOLD: Cough_Audio_Coswera
num_workers: 0 0 0
Train batch: (32, 3, 224, 224) | first labels: [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1]
Train counts [healthy,cancer]: [603, 21]
Sampler class weights: [0.5174129605293274, 14.857142448425293]
Sizes -> train/val/test: 624 178 284


In [13]:
# LOSO (hold-out Coswera) — EfficientNet-B0 training on this fold
import os, numpy as np, torch
from torch import nn
from torch.optim import AdamW
from torchvision import models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

assert 'train_loader' in globals() and 'val_loader' in globals() and 'test_loader' in globals()
assert 'train_ds' in globals()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# fresh EfficientNet-B0 (ImageNet weights) -> 2 classes
effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
model = effnet.to(device)

# class-weighted CE from THIS fold's TRAIN
LABEL2ID = {"healthy":0, "cancer":1}
y_train = train_ds.df["label"].map(LABEL2ID).values
cnts    = np.bincount(y_train, minlength=2)
class_w = torch.tensor([cnts.sum()/(2*max(1,cnts[0])),
                        cnts.sum()/(2*max(1,cnts[1]))],
                       dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_w)

opt   = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train)
    losses, ys, ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        probs = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
        ys.extend(yb.detach().cpu().numpy().tolist())
        ps.extend(probs.tolist())
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps >= 0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(ys, ps)
    except ValueError: auc = float('nan')
    return float(np.mean(losses)), acc, prec, rec, f1, auc

EPOCHS, PATIENCE = 20, 4
best_auc, best_epoch, best_state, no_improve = -1.0, -1, None, 0

OUTDIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/effnet"
os.makedirs(f"{OUTDIR}/checkpoints", exist_ok=True)
SAVE_PATH = f"{OUTDIR}/checkpoints/efficientnet_b0_best.pt"

for ep in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | "
          f"val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1] > best_auc:
        best_auc, best_epoch = vl[-1], ep
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        torch.save(best_state, SAVE_PATH)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {ep} (best val AUC at {best_epoch}: {best_auc:.3f})")
            break

print(f"\nBest epoch: {best_epoch} | Best val AUC: {best_auc:.3f}")
print("Saved:", SAVE_PATH)

# quick TEST preview at τ=0.5 (we will tune τ on VAL in next step)
if best_state is not None:
    model.load_state_dict(best_state, strict=False)
te = run_epoch(test_loader, train=False)
print("\nTEST @0.5 preview -> loss {:.4f} | acc {:.3f} | prec {:.3f} | rec {:.3f} | f1 {:.3f} | auc {:.3f}".format(*te))


Epoch 01 | train loss 0.2506 f1 0.917 auc 0.983 | val loss 0.2379 f1 0.600 auc 0.991
Epoch 02 | train loss 0.0210 f1 0.988 auc 0.999 | val loss 0.1238 f1 0.706 auc 0.994
Epoch 03 | train loss 0.0145 f1 0.979 auc 0.997 | val loss 0.0686 f1 0.706 auc 0.999
Epoch 04 | train loss 0.0081 f1 0.988 auc 0.999 | val loss 0.0554 f1 0.800 auc 1.000
Epoch 05 | train loss 0.0051 f1 0.991 auc 1.000 | val loss 0.0562 f1 0.800 auc 1.000
Epoch 06 | train loss 0.0053 f1 0.993 auc 1.000 | val loss 0.0444 f1 0.800 auc 0.999
Epoch 07 | train loss 0.0024 f1 0.998 auc 1.000 | val loss 0.0384 f1 0.923 auc 0.999
Epoch 08 | train loss 0.0036 f1 0.993 auc 1.000 | val loss 0.0444 f1 0.923 auc 0.999
Early stopping at epoch 8 (best val AUC at 4: 1.000)

Best epoch: 4 | Best val AUC: 1.000
Saved: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/effnet/checkpoints/efficientnet_b0_best.pt

TEST @0.5 preview -> loss 0.0443 | acc 1.000 | prec 1.000 | rec 1.000 | f1

In [14]:
# LOSO (hold-out Coswera) — threshold on VAL → apply to TEST + save
import os, numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score

FOLD_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera"
CKPT     = f"{FOLD_DIR}/effnet/checkpoints/efficientnet_b0_best.pt"
OUTCSV   = f"{FOLD_DIR}/effnet/predictions_val_test.csv"
TAU_TXT  = f"{FOLD_DIR}/effnet/operating_threshold.txt"
os.makedirs(os.path.dirname(OUTCSV), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# use the model from training step
from torchvision import models
from torch import nn
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
state = torch.load(CKPT, map_location=device)
model.load_state_dict(state, strict=False)
model = model.to(device).eval()

def collect(loader, dataset):
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pr = torch.softmax(model(xb), dim=1)[:,1].cpu().numpy()
            probs.extend(pr); ys.extend(yb.numpy())
    df = dataset.df.copy()
    df["y_true"] = ys
    df["prob_cancer"] = probs
    return df

val_df  = collect(val_loader,  val_ds)
test_df = collect(test_loader, test_ds)

def metrics_at(y, p, tau):
    yhat = (p >= tau).astype(int)
    acc = accuracy_score(y, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    return acc, prec, rec, f1

def choose_tau_unique(y, p, prec_target=0.98):
    ps = np.unique(p)
    mids = [(ps[i]+ps[i+1])/2.0 for i in range(len(ps)-1)]
    cands = np.array([0.0] + mids + [1.0])
    best = None
    for t in cands:
        acc,pr,re,f1 = metrics_at(y,p,t)
        if pr >= prec_target:
            item = (re, f1, t, acc, pr)
            if best is None or item > best: best = item
    if best is not None:
        re,f1,t,acc,pr = best
        return float(t), {"policy":"prec>=0.98","acc":acc,"prec":pr,"rec":re,"f1":f1}
    # fallback: best F1
    best = (-1,None,None,None)
    for t in cands:
        acc,pr,re,f1 = metrics_at(y,p,t)
        if f1 > best[0]: best = (f1,t,(acc,pr,re))
    f1,t,(acc,pr,re) = best
    return float(t), {"policy":"bestF1","acc":acc,"prec":pr,"rec":re,"f1":f1}

# choose τ* on VAL
yv, pv = val_df["y_true"].values,  val_df["prob_cancer"].values
tau_star, val_sum = choose_tau_unique(yv, pv, prec_target=0.98)
print(f"Chosen τ*={tau_star:.6f}  policy={val_sum['policy']} | VAL acc={val_sum['acc']:.3f} prec={val_sum['prec']:.3f} rec={val_sum['rec']:.3f} f1={val_sum['f1']:.3f}")

# apply to TEST
yt, pt = test_df["y_true"].values, test_df["prob_cancer"].values
acc,pr,re,f1 = metrics_at(yt, pt, tau_star)
auc = roc_auc_score(yt, pt)
cm  = confusion_matrix(yt, (pt>=tau_star).astype(int), labels=[0,1])
print(f"[TEST @ τ*] acc={acc:.3f} prec={pr:.3f} rec={re:.3f} f1={f1:.3f} auc={auc:.3f}")
print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)

# save predictions + τ*
pd.concat([val_df.assign(split='val'), test_df.assign(split='test')]).to_csv(OUTCSV, index=False)
with open(TAU_TXT,"w") as f: f.write(f"{tau_star:.6f}\npolicy={val_sum['policy']}\n")
print(f"\n✅ Saved predictions to: {OUTCSV}")
print(f"τ* written to: {TAU_TXT}")


Chosen τ*=0.992801  policy=prec>=0.98 | VAL acc=1.000 prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[281   0]
 [  0   3]]

✅ Saved predictions to: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/effnet/predictions_val_test.csv
τ* written to: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/effnet/operating_threshold.txt


In [15]:
# LOSO (hold-out COUGHVID) — rebuild loaders
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T

FOLD_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID"
TRAIN_CSV, VAL_CSV, TEST_CSV = [f"{FOLD_DIR}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]), "LOSO CSVs not found."

LABEL2ID = {"healthy":0, "cancer":1}

class SpecAug2D(torch.nn.Module):
    def __init__(self, T_mask=30, F_mask=20, num_T=2, num_F=2, p=0.7):
        super().__init__(); self.T_mask=T_mask; self.F_mask=F_mask; self.num_T=num_T; self.num_F=num_F; self.p=p
    def forward(self, x):
        if not self.training or torch.rand(())>self.p: return x
        c,H,W = x.shape
        for _ in range(self.num_T):
            w = int(torch.randint(0, self.T_mask+1, (1,)))
            if 0<w<W:
                t0 = int(torch.randint(0, W-w+1, (1,))); x[:, :, t0:t0+w] = 0.0
        for _ in range(self.num_F):
            h = int(torch.randint(0, self.F_mask+1, (1,)))
            if 0<h<H:
                f0 = int(torch.randint(0, H-h+1, (1,))); x[:, f0:f0+h, :] = 0.0
        return x

def train_tf(minority=False):
    spec = SpecAug2D(T_mask=40 if minority else 30, F_mask=24 if minority else 20, p=0.9 if minority else 0.7)
    return T.Compose([
        T.Lambda(lambda p: p.convert("L")),
        T.ToTensor(),
        spec,
        T.Lambda(lambda t: t.repeat(3,1,1)),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class CSVImageDS(Dataset):
    def __init__(self, csv_path, tf_h, tf_c):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df; self.tf_h=tf_h; self.tf_c=tf_c
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = LABEL2ID[str(r["label"]).lower().strip()]
        x = Image.open(r["filepath"])
        tf = self.tf_c if y==1 else self.tf_h
        return tf(x), torch.tensor(y, dtype=torch.long)

train_ds = CSVImageDS(TRAIN_CSV, tf_h=train_tf(False), tf_c=train_tf(True))
val_ds   = CSVImageDS(VAL_CSV,   tf_h=eval_tf,         tf_c=eval_tf)
test_ds  = CSVImageDS(TEST_CSV,  tf_h=eval_tf,         tf_c=eval_tf)

y_train = train_ds.df["label"].map(LABEL2ID).values
counts  = np.bincount(y_train, minlength=2).astype(np.float32)
class_sampling_w = (counts.sum() / (2.0*np.maximum(counts,1.0))).astype(np.float32)
weights = class_sampling_w[y_train]
sampler = WeightedRandomSampler(weights.tolist(), num_samples=len(weights), replacement=True)

BATCH=32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False)

xb, yb = next(iter(train_loader))
print("FOLD:", FOLD_DIR.split("/")[-1])
print("num_workers:", train_loader.num_workers, val_loader.num_workers, test_loader.num_workers)
print("Train batch:", tuple(xb.shape), "| first labels:", yb[:12].tolist())
print("Train counts [healthy,cancer]:", counts.astype(int).tolist())
print("Sampler class weights:", class_sampling_w.tolist())
print("Sizes -> train/val/test:", len(train_ds), len(val_ds), len(test_ds))


FOLD: Cough_Audio_COUGHVID
num_workers: 0 0 0
Train batch: (32, 3, 224, 224) | first labels: [0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1]
Train counts [healthy,cancer]: [1983, 21]
Sampler class weights: [0.5052949786186218, 47.71428680419922]
Sizes -> train/val/test: 2004 573 87


In [16]:
# LOSO (hold-out COUGHVID) — EfficientNet-B0 training on this fold
import os, numpy as np, torch
from torch import nn
from torch.optim import AdamW
from torchvision import models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

assert 'train_loader' in globals() and 'val_loader' in globals() and 'test_loader' in globals()
assert 'train_ds' in globals()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# fresh EfficientNet-B0 (ImageNet weights) -> 2 classes
effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
model = effnet.to(device)

# class-weighted CE from THIS fold's TRAIN
LABEL2ID = {"healthy":0, "cancer":1}
y_train = train_ds.df["label"].map(LABEL2ID).values
cnts    = np.bincount(y_train, minlength=2)
class_w = torch.tensor([cnts.sum()/(2*max(1,cnts[0])),
                        cnts.sum()/(2*max(1,cnts[1]))],
                       dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_w)

opt   = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train)
    losses, ys, ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        probs = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
        ys.extend(yb.detach().cpu().numpy().tolist())
        ps.extend(probs.tolist())
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps >= 0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(ys, ps)
    except ValueError: auc = float('nan')
    return float(np.mean(losses)), acc, prec, rec, f1, auc

EPOCHS, PATIENCE = 20, 4
best_auc, best_epoch, best_state, no_improve = -1.0, -1, None, 0

OUTDIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/effnet"
os.makedirs(f"{OUTDIR}/checkpoints", exist_ok=True)
SAVE_PATH = f"{OUTDIR}/checkpoints/efficientnet_b0_best.pt"

for ep in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | "
          f"val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1] > best_auc:
        best_auc, best_epoch = vl[-1], ep
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        torch.save(best_state, SAVE_PATH)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {ep} (best val AUC at {best_epoch}: {best_auc:.3f})")
            break

print(f"\nBest epoch: {best_epoch} | Best val AUC: {best_auc:.3f}")
print("Saved:", SAVE_PATH)

# quick TEST preview at τ=0.5
if best_state is not None:
    model.load_state_dict(best_state, strict=False)
te = run_epoch(test_loader, train=False)
print("\nTEST @0.5 preview -> loss {:.4f} | acc {:.3f} | prec {:.3f} | rec {:.3f} | f1 {:.3f} | auc {:.3f}".format(*te))


Epoch 01 | train loss 0.0705 f1 0.976 auc 0.997 | val loss 0.0869 f1 0.706 auc 0.999
Epoch 02 | train loss 0.0026 f1 0.989 auc 1.000 | val loss 0.0634 f1 0.706 auc 0.999
Epoch 03 | train loss 0.0013 f1 0.997 auc 1.000 | val loss 0.0406 f1 0.706 auc 0.999
Epoch 04 | train loss 0.0012 f1 0.996 auc 0.999 | val loss 0.0306 f1 0.750 auc 1.000
Epoch 05 | train loss 0.0006 f1 0.998 auc 1.000 | val loss 0.0235 f1 0.750 auc 1.000
Epoch 06 | train loss 0.0005 f1 0.998 auc 1.000 | val loss 0.0161 f1 0.923 auc 1.000
Epoch 07 | train loss 0.0006 f1 0.998 auc 1.000 | val loss 0.0166 f1 0.923 auc 1.000
Epoch 08 | train loss 0.0004 f1 0.999 auc 1.000 | val loss 0.0124 f1 0.923 auc 1.000
Early stopping at epoch 8 (best val AUC at 4: 1.000)

Best epoch: 4 | Best val AUC: 1.000
Saved: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/effnet/checkpoints/efficientnet_b0_best.pt

TEST @0.5 preview -> loss 0.0269 | acc 1.000 | prec 1.000 | rec 1.000 | f

In [17]:
# LOSO (hold-out COUGHVID) — threshold on VAL → apply to TEST + save
import os, numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from torchvision import models
from torch import nn

FOLD_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID"
CKPT     = f"{FOLD_DIR}/effnet/checkpoints/efficientnet_b0_best.pt"
OUTCSV   = f"{FOLD_DIR}/effnet/predictions_val_test.csv"
TAU_TXT  = f"{FOLD_DIR}/effnet/operating_threshold.txt"
os.makedirs(os.path.dirname(OUTCSV), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
state = torch.load(CKPT, map_location=device)
model.load_state_dict(state, strict=False)
model = model.to(device).eval()

def collect(loader, dataset):
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pr = torch.softmax(model(xb), dim=1)[:,1].cpu().numpy()
            probs.extend(pr); ys.extend(yb.numpy())
    df = dataset.df.copy()
    df["y_true"] = ys
    df["prob_cancer"] = probs
    return df

val_df  = collect(val_loader,  val_ds)
test_df = collect(test_loader, test_ds)

def metrics_at(y, p, tau):
    yhat = (p >= tau).astype(int)
    acc = accuracy_score(y, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    return acc, prec, rec, f1

def choose_tau_unique(y, p, prec_target=0.98):
    ps = np.unique(p)
    mids = [(ps[i]+ps[i+1])/2.0 for i in range(len(ps)-1)]
    cands = np.array([0.0] + mids + [1.0])
    best = None
    for t in cands:
        acc,pr,re,f1 = metrics_at(y,p,t)
        if pr >= prec_target:
            item = (re, f1, t, acc, pr)
            if best is None or item > best: best = item
    if best is not None:
        re,f1,t,acc,pr = best
        return float(t), {"policy":"prec>=0.98","acc":acc,"prec":pr,"rec":re,"f1":f1}
    # fallback: best F1
    best = (-1,None,None,None)
    for t in cands:
        acc,pr,re,f1 = metrics_at(y,p,t)
        if f1 > best[0]: best = (f1,t,(acc,pr,re))
    f1,t,(acc,pr,re) = best
    return float(t), {"policy":"bestF1","acc":acc,"prec":pr,"rec":re,"f1":f1}

# choose τ* on VAL
yv, pv = val_df["y_true"].values,  val_df["prob_cancer"].values
tau_star, val_sum = choose_tau_unique(yv, pv, prec_target=0.98)
print(f"Chosen τ*={tau_star:.6f}  policy={val_sum['policy']} | VAL acc={val_sum['acc']:.3f} prec={val_sum['prec']:.3f} rec={val_sum['rec']:.3f} f1={val_sum['f1']:.3f}")

# apply to TEST
yt, pt = test_df["y_true"].values, test_df["prob_cancer"].values
acc,pr,re,f1 = metrics_at(yt, pt, tau_star)
auc = roc_auc_score(yt, pt)
cm  = confusion_matrix(yt, (pt>=tau_star).astype(int), labels=[0,1])
print(f"[TEST @ τ*] acc={acc:.3f} prec={pr:.3f} rec={re:.3f} f1={f1:.3f} auc={auc:.3f}")
print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)

pd.concat([val_df.assign(split='val'), test_df.assign(split='test')]).to_csv(OUTCSV, index=False)
with open(TAU_TXT,"w") as f: f.write(f"{tau_star:.6f}\npolicy={val_sum['policy']}\n")
print(f"\n✅ Saved predictions to: {OUTCSV}")
print(f"τ* written to: {TAU_TXT}")


Chosen τ*=0.999645  policy=prec>=0.98 | VAL acc=1.000 prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM (rows=true [healthy,cancer], cols=pred):
 [[84  0]
 [ 0  3]]

✅ Saved predictions to: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/effnet/predictions_val_test.csv
τ* written to: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/effnet/operating_threshold.txt


In [18]:
# Near-duplicate audit across both LOSO folds using average-hash (aHash)
import os, pandas as pd, numpy as np
from PIL import Image

LOSO_ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
FOLDS = ["Cough_Audio_Coswera", "Cough_Audio_COUGHVID"]
HASH_SIZE = 8
NEAR_THRESH = 5  # Hamming distance threshold for near-duplicate

def average_hash(path, hash_size=HASH_SIZE):
    try:
        img = Image.open(path).convert("L").resize((hash_size, hash_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32)
        m = arr.mean()
        bits = (arr > m).astype(np.uint8).flatten()
        # pack into 64-bit integer
        h = 0
        for b in bits: h = (h << 1) | int(b)
        return h
    except Exception as e:
        return None

def hamming(a, b):
    return (int(a) ^ int(b)).bit_count()

def load_split(folder):
    paths = {}
    for split in ("train","val","test"):
        p = os.path.join(LOSO_ROOT, folder, f"{split}.csv")
        df = pd.read_csv(p)
        df = df[df["filepath"].apply(lambda x: isinstance(x,str) and os.path.exists(x))].copy()
        paths[split] = df
    return paths

def audit_fold(name):
    D = load_split(name)
    trv = pd.concat([D["train"], D["val"]], ignore_index=True)
    te  = D["test"]
    set_trv = set(trv["filepath"])
    set_te  = set(te["filepath"])

    # 1) exact filepath overlaps
    exact_overlap = sorted(set_trv & set_te)

    # 2) compute hashes
    all_paths = list(set_trv | set_te)
    hash_map = {}
    for p in all_paths:
        hh = average_hash(p)
        if hh is not None:
            hash_map[p] = hh
    # subsets with available hashes
    trv_h = trv[trv["filepath"].isin(hash_map)]
    te_h  = te[te["filepath"].isin(hash_map)]

    # 3) nearest neighbor Hamming from each TEST to TRAIN/VAL
    dup_exact, dup_near = [], []
    for _, r in te_h.iterrows():
        pt = r["filepath"]; ht = hash_map[pt]
        # linear scan (small enough)
        best = (1e9, None)
        for _, s in trv_h.iterrows():
            ps = s["filepath"]; hs = hash_map[ps]
            d = hamming(ht, hs)
            if d < best[0]:
                best = (d, ps)
        dmin, pbest = best
        if dmin == 0:
            dup_exact.append((pt, pbest, dmin,
                              r["label"], r["source"],
                              trv_h.loc[trv_h["filepath"]==pbest, "label"].values[0],
                              trv_h.loc[trv_h["filepath"]==pbest, "source"].values[0]))
        elif dmin <= NEAR_THRESH:
            dup_near.append((pt, pbest, dmin,
                             r["label"], r["source"],
                             trv_h.loc[trv_h["filepath"]==pbest, "label"].values[0],
                             trv_h.loc[trv_h["filepath"]==pbest, "source"].values[0]))

    print("\n" + "="*80)
    print(f"FOLD: {name}")
    print(f"Train+Val n={len(trv)}, Test n={len(te)}")
    print(f"Exact filepath overlaps (should be 0): {len(exact_overlap)}")
    print(f"Hashable Train+Val n={len(trv_h)}, Hashable Test n={len(te_h)}")
    print(f"Pixel-duplicate (aHash dist=0) test→train/val: {len(dup_exact)}")
    print(f"Near-duplicates (aHash dist≤{NEAR_THRESH}) test→train/val: {len(dup_near)}")

    # show a few examples
    def show_rows(rows, title, k=5):
        if not rows:
            print(f"{title}: none")
            return
        print(f"{title} (showing up to {k}):")
        for i,(pt, pbest, d, yt, st, ytr, strc) in enumerate(rows[:k], 1):
            print(f"  [{i}] d={d} | TEST[{yt},{st}] -> TRAIN/VAL[{ytr},{strc}]")
            print(f"      test:  {pt}")
            print(f"      match: {pbest}")
    show_rows(dup_exact, "  Exact pixel-duplicate pairs")
    show_rows(dup_near,  "  Near-duplicate pairs")

for fold in FOLDS:
    audit_fold(fold)

print("\n✅ Duplicate audit complete. If any counts above are >0, we likely have leakage.")



FOLD: Cough_Audio_Coswera
Train+Val n=802, Test n=284
Exact filepath overlaps (should be 0): 0
Hashable Train+Val n=802, Hashable Test n=284
Pixel-duplicate (aHash dist=0) test→train/val: 6
Near-duplicates (aHash dist≤5) test→train/val: 98
  Exact pixel-duplicate pairs (showing up to 5):
  [1] d=0 | TEST[healthy,Cough_Audio_Coswera] -> TRAIN/VAL[healthy,Cough_Audio_COUGHVID]
      test:  /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/Cough_Audio_Coswera/healthy/aNOpaVGqF7PiG5n1ln5B1VJnmCO2_cough-heavy.png
      match: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/Cough_Audio_COUGHVID/healthy2/34b7eed3-4f5a-4ae6-9c61-56e913199bd4.png
  [2] d=0 | TEST[healthy,Cough_Audio_Coswera] -> TRAIN/VAL[healthy,Cough_Audio_COUGHVID]
      test:  /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Le

In [19]:
# Deduplicate LOSO splits by removing exact pixel duplicates (aHash=0) from TRAIN/VAL if they appear in TEST
import os, pandas as pd, numpy as np
from PIL import Image

LOSO_ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO"
FOLDS = ["Cough_Audio_Coswera", "Cough_Audio_COUGHVID"]
HASH_SIZE = 8

def average_hash(path, hash_size=HASH_SIZE):
    try:
        img = Image.open(path).convert("L").resize((hash_size, hash_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32)
        m = arr.mean()
        bits = (arr > m).astype(np.uint8).flatten()
        h=0
        for b in bits: h=(h<<1)|int(b)
        return h
    except Exception:
        return None

def load_split(folder):
    def _read(split):
        p = os.path.join(LOSO_ROOT, folder, f"{split}.csv")
        df = pd.read_csv(p)
        df = df[df["filepath"].apply(lambda x: isinstance(x,str) and os.path.exists(x))].copy()
        return df
    return {"train":_read("train"), "val":_read("val"), "test":_read("test")}

def write_split(folder, tag, dfs):
    outdir = os.path.join(LOSO_ROOT, folder, tag)
    os.makedirs(outdir, exist_ok=True)
    for k,df in dfs.items():
        df.to_csv(os.path.join(outdir, f"{k}.csv"), index=False)
    return outdir

for fold in FOLDS:
    D   = load_split(fold)
    trv = pd.concat([D["train"].assign(split="train"), D["val"].assign(split="val")], ignore_index=True)
    te  = D["test"].copy()

    # build TEST hash set
    te["ahash"] = te["filepath"].apply(average_hash)
    te_h = te.dropna(subset=["ahash"])
    test_hashes = set(te_h["ahash"].tolist())

    # hash train/val
    trv["ahash"] = trv["filepath"].apply(average_hash)
    trv_h = trv.dropna(subset=["ahash"])

    # mark leaks: trv items whose ahash equals any test ahash
    leak_mask = trv_h["ahash"].isin(test_hashes)
    leaks = trv_h[leak_mask]
    print("\n" + "="*72)
    print(f"FOLD: {fold}")
    print(f"Train+Val total: {len(trv)} | hashable: {len(trv_h)}")
    print(f"Test total:      {len(te)}  | hashable: {len(te_h)}")
    print(f"Exact duplicates (aHash=0) crossing TEST↔(TRAIN/VAL): {leaks.shape[0]}")

    # drop duplicates from TRAIN/VAL (keep TEST intact)
    keep = trv_h[~leak_mask].copy()
    # reconstruct train/val
    train_new = keep[keep["split"]=="train"].drop(columns=["split","ahash"])
    val_new   = keep[keep["split"]=="val"].drop(columns=["split","ahash"])
    test_new  = te.drop(columns=["ahash"])

    outdir = write_split(fold, "dedup_exact", {"train":train_new, "val":val_new, "test":test_new})
    # report per-split deltas
    tdrop = len(D["train"]) - len(train_new)
    vdrop = len(D["val"])   - len(val_new)
    print(f"Saved de-duplicated splits -> {outdir}")
    print(f"Dropped from TRAIN: {tdrop}  | from VAL: {vdrop}  | TEST unchanged.")



FOLD: Cough_Audio_Coswera
Train+Val total: 802 | hashable: 802
Test total:      284  | hashable: 284
Exact duplicates (aHash=0) crossing TEST↔(TRAIN/VAL): 20
Saved de-duplicated splits -> /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/dedup_exact
Dropped from TRAIN: 16  | from VAL: 4  | TEST unchanged.

FOLD: Cough_Audio_COUGHVID
Train+Val total: 2577 | hashable: 2577
Test total:      87  | hashable: 87
Exact duplicates (aHash=0) crossing TEST↔(TRAIN/VAL): 51
Saved de-duplicated splits -> /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/dedup_exact
Dropped from TRAIN: 37  | from VAL: 14  | TEST unchanged.


In [20]:
# Rebuild loaders from dedup_exact for Coswera-held-out, then train+eval
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T
from torch import nn
from torchvision import models
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

FOLD = "Cough_Audio_Coswera"
DED = f"/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/{FOLD}/dedup_exact"
TRAIN_CSV, VAL_CSV, TEST_CSV = [f"{DED}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [TRAIN_CSV, VAL_CSV, TEST_CSV])

LABEL2ID = {"healthy":0,"cancer":1}
eval_tf = T.Compose([T.Lambda(lambda p: p.convert("L")), T.ToTensor(),
                     T.Lambda(lambda t: t.repeat(3,1,1)),
                     T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
class DS(Dataset):
    def __init__(self, csv, tf): 
        df = pd.read_csv(csv); df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))]
        self.df=df.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; x=Image.open(r["filepath"]); y=LABEL2ID[str(r["label"]).lower()]
        return self.tf(x), torch.tensor(y)

train_ds, val_ds, test_ds = DS(TRAIN_CSV, eval_tf), DS(VAL_CSV, eval_tf), DS(TEST_CSV, eval_tf)
ytr = train_ds.df["label"].map(LABEL2ID).values; cnts = np.bincount(ytr, minlength=2).astype(np.float32)
w = (cnts.sum()/(2*np.maximum(cnts,1))).astype(np.float32); sampler = WeightedRandomSampler(w[ytr].tolist(), num_samples=len(ytr), replacement=True)

BATCH=32
train_loader = DataLoader(train_ds,batch_size=BATCH,sampler=sampler,num_workers=0)
val_loader   = DataLoader(val_ds,  batch_size=BATCH,shuffle=False,num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH,shuffle=False,num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)
class_w = torch.tensor([cnts.sum()/(2*max(1,cnts[0])), cnts.sum()/(2*max(1,cnts[1]))], dtype=torch.float32, device=device)
crit = nn.CrossEntropyLoss(weight=class_w)
opt  = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched= torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train); losses=[]; ys=[]; ps=[]
    for xb,yb in loader:
        xb,yb=xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            lg=model(xb); loss=crit(lg,yb)
            if train: opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        losses.append(loss.item()); pr=torch.softmax(lg,1)[:,1].detach().cpu().numpy()
        ys+=yb.cpu().numpy().tolist(); ps+=pr.tolist()
    import numpy as np
    ys,ps=np.array(ys),np.array(ps); yhat=(ps>=0.5).astype(int)
    acc=accuracy_score(ys,yhat); pr,rc,f1,_=precision_recall_fscore_support(ys,yhat,average='binary',zero_division=0)
    try: auc=roc_auc_score(ys,ps)
    except: auc=float('nan')
    return float(np.mean(losses)), acc, pr, rc, f1, auc

best_auc=-1; best=None; EPOCHS=20; PAT=4
CKPT=f"{DED}/effnet_retrain_best.pt"; os.makedirs(os.path.dirname(CKPT), exist_ok=True)
for ep in range(1,EPOCHS+1):
    tr=run_epoch(train_loader,True); vl=run_epoch(val_loader,False); sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1]>best_auc: best_auc=vl[-1]; best={k:v.cpu() for k,v in model.state_dict().items()}
    else:
        PAT-=1
        if PAT<=0: print(f"Early stop at {ep}, best val AUC {best_auc:.3f}"); break
if best: 
    torch.save(best, CKPT); model.load_state_dict(best, strict=False)

# τ* on VAL (precision≥0.98 else best-F1)
def choose_tau(y,p, prec_target=0.98):
    ps=np.unique(p); mids=[(ps[i]+ps[i+1])/2 for i in range(len(ps)-1)]
    cands=np.array([0.0]+mids+[1.0]); pick=None
    from sklearn.metrics import precision_recall_fscore_support
    for t in cands:
        yhat=(p>=t).astype(int); pr,rc,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if pr>=prec_target: item=(rc,f1,t,pr); 
        else: item=None
        if item and (pick is None or item>pick): pick=item
    if pick: rc,f1,t,pr=pick; return float(t),"prec>=0.98",pr,rc,f1
    best=(-1,None)
    for t in cands:
        yhat=(p>=t).astype(int); pr,rc,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if f1>best[0]: best=(f1,(t,pr,rc))
    f1,(t,pr,rc)=best; return float(t),"bestF1",pr,rc,f1

def collect(loader, ds):
    probs=[]; ys=[]
    with torch.no_grad():
        for xb,yb in loader:
            pr=torch.softmax(model(xb.to(device)),1)[:,1].cpu().numpy()
            probs+=pr.tolist(); ys+=yb.numpy().tolist()
    df=ds.df.copy(); df["y_true"]=ys; df["prob_cancer"]=probs; return df

vd=collect(val_loader,val_ds); td=collect(test_loader,test_ds)
tau,pol,pr,rc,f1=choose_tau(vd["y_true"].values, vd["prob_cancer"].values, prec_target=0.98)
from sklearn.metrics import roc_auc_score, confusion_matrix
yt,pt=td["y_true"].values, td["prob_cancer"].values
yhat=(pt>=tau).astype(int)
acc=accuracy_score(yt,yhat); prt, rct, f1t, _ = precision_recall_fscore_support(yt,yhat,average='binary',zero_division=0)
auc=roc_auc_score(yt,pt); cm=confusion_matrix(yt,yhat,labels=[0,1])
print(f"\nVAL τ*={tau:.6f} policy={pol} | VAL prec={pr:.3f} rec={rc:.3f} f1={f1:.3f}")
print(f"[TEST @ τ*] acc={acc:.3f} prec={prt:.3f} rec={rct:.3f} f1={f1t:.3f} auc={auc:.3f}")
print('CM:\n', cm)


Epoch 01 | train loss 0.2639 f1 0.944 auc 0.989 | val loss 0.2515 f1 0.667 auc 0.987
Epoch 02 | train loss 0.0140 f1 0.992 auc 0.998 | val loss 0.1138 f1 0.667 auc 0.998
Epoch 03 | train loss 0.0054 f1 0.994 auc 1.000 | val loss 0.0626 f1 0.714 auc 0.999
Epoch 04 | train loss 0.0050 f1 0.986 auc 1.000 | val loss 0.0541 f1 0.714 auc 1.000
Epoch 05 | train loss 0.0031 f1 0.994 auc 1.000 | val loss 0.0396 f1 0.909 auc 1.000
Epoch 06 | train loss 0.0020 f1 0.998 auc 1.000 | val loss 0.0202 f1 0.909 auc 1.000
Epoch 07 | train loss 0.0019 f1 0.998 auc 1.000 | val loss 0.0175 f1 0.909 auc 1.000
Epoch 08 | train loss 0.0020 f1 1.000 auc 1.000 | val loss 0.0160 f1 0.909 auc 1.000
Early stop at 8, best val AUC 1.000

VAL τ*=0.982648 policy=prec>=0.98 | VAL prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM:
 [[281   0]
 [  0   3]]


In [21]:
# De-duplicate splits against TEST (and optionally VAL against TRAIN)
import os, numpy as np, pandas as pd
from PIL import Image

def _exists(p): return isinstance(p, str) and os.path.exists(p)

def average_hash(path, hash_size=8):
    try:
        img = Image.open(path).convert("L").resize((hash_size, hash_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32); m = arr.mean()
        bits = (arr > m).astype(np.uint8).flatten()
        h = 0
        for b in bits: h = (h << 1) | int(b)
        return h
    except Exception:
        return None

def dhash(path, hash_size=8):
    try:
        img = Image.open(path).convert("L").resize((hash_size+1, hash_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.int16)
        diff = (arr[:,1:] > arr[:,:-1]).astype(np.uint8).flatten()
        h = 0
        for b in diff: h = (h << 1) | int(b)
        return h
    except Exception:
        return None

def hamming(a, b): return (int(a) ^ int(b)).bit_count()

def _read_split(split_dir, name):
    p = os.path.join(split_dir, f"{name}.csv")
    df = pd.read_csv(p)
    return df[df["filepath"].apply(_exists)].reset_index(drop=True)

def _hash_df(df):
    df = df.copy()
    df["ahash"] = df["filepath"].apply(average_hash)
    df["dhash"] = df["filepath"].apply(dhash)
    return df.dropna(subset=["ahash","dhash"]).reset_index(drop=True)

def build_dedup_splits(split_dir,
                       out_subdir="dedup_near",
                       mode="near",          # "exact" or "near"
                       ah_t=5, dh_t=5,       # thresholds for "near"
                       dedup_val_against_train=True):
    assert mode in ("exact","near")
    assert os.path.exists(split_dir), f"Not found: {split_dir}"
    train = _read_split(split_dir, "train")
    val   = _read_split(split_dir, "val")
    test  = _read_split(split_dir, "test")

    trv = pd.concat([train.assign(split="train"),
                     val.assign(split="val")], ignore_index=True)

    # hash all
    te_h  = _hash_df(test)
    trv_h = _hash_df(trv)

    # build masks: drop trv items that collide with ANY test item
    drop_mask = np.zeros(len(trv_h), dtype=bool)
    te_ah, te_dh = te_h["ahash"].values, te_h["dhash"].values
    for i, (a, d) in enumerate(zip(trv_h["ahash"].values, trv_h["dhash"].values)):
        if mode == "exact":
            # exact duplicate if either hash exactly matches
            cond = (np.array([hamming(a, x) for x in te_ah]) == 0) & \
                   (np.array([hamming(d, x) for x in te_dh]) == 0)
            drop_mask[i] = cond.any()
        else:
            cond = (np.array([hamming(a, x) for x in te_ah]) <= ah_t) & \
                   (np.array([hamming(d, x) for x in te_dh]) <= dh_t)
            drop_mask[i] = cond.any()

    leaks = trv_h[drop_mask]
    keep  = trv_h[~drop_mask]

    # (optional) also drop VAL items that are near-duplicates of TRAIN items
    if dedup_val_against_train:
        keep_train = keep[keep["split"]=="train"].copy()
        keep_val   = keep[keep["split"]=="val"].copy()
        if len(keep_train) and len(keep_val):
            ta, td = keep_train["ahash"].values, keep_train["dhash"].values
            drop_val = np.zeros(len(keep_val), dtype=bool)
            for i,(a,d) in enumerate(zip(keep_val["ahash"].values, keep_val["dhash"].values)):
                if mode == "exact":
                    cond = (np.array([hamming(a, x) for x in ta]) == 0) & \
                           (np.array([hamming(d, x) for x in td]) == 0)
                    drop_val[i] = cond.any()
                else:
                    cond = (np.array([hamming(a, x) for x in ta]) <= ah_t) & \
                           (np.array([hamming(d, x) for x in td]) <= dh_t)
                    drop_val[i] = cond.any()
            val_dups = keep_val[drop_val]
            keep_val = keep_val[~drop_val]
        else:
            val_dups = keep_val.iloc[0:0]
        # reassemble keep
        keep = pd.concat([keep_train, keep_val], ignore_index=True)
    else:
        val_dups = trv_h.iloc[0:0]  # empty

    # strip helper cols and split back
    def _strip(df): return df.drop(columns=["ahash","dhash"])
    train_new = _strip(keep[keep["split"]=="train"]).drop(columns=["split"])
    val_new   = _strip(keep[keep["split"]=="val"]).drop(columns=["split"])
    test_new  = test.copy()

    outdir = os.path.join(split_dir, out_subdir)
    os.makedirs(outdir, exist_ok=True)
    train_new.to_csv(os.path.join(outdir, "train.csv"), index=False)
    val_new.to_csv(  os.path.join(outdir, "val.csv"),   index=False)
    test_new.to_csv( os.path.join(outdir, "test.csv"),  index=False)

    print(f"\n=== Dedup @ {split_dir} → {out_subdir} ({mode}) ===")
    print(f"Train/Val/Test in -> {len(train)}/{len(val)}/{len(test)}")
    print(f"Removed vs TEST  -> {len(leaks)} from TRAIN/VAL")
    if dedup_val_against_train:
        print(f"Removed VAL vs TRAIN (near/ exact) -> {len(val_dups)}")
    print(f"Out sizes        -> {len(train_new)}/{len(val_new)}/{len(test_new)}")
    return outdir


In [22]:
# 1) LOSO folds
build_dedup_splits(
    "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera",
    out_subdir="dedup_near", mode="near", ah_t=5, dh_t=5
)
build_dedup_splits(
    "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID",
    out_subdir="dedup_near", mode="near", ah_t=5, dh_t=5
)

# (optional) your base single-split directory too:
# build_dedup_splits("<...>/metadata/splits", out_subdir="dedup_near", mode="near", ah_t=5, dh_t=5)



=== Dedup @ /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera → dedup_near (near) ===
Train/Val/Test in -> 624/178/284
Removed vs TEST  -> 25 from TRAIN/VAL
Removed VAL vs TRAIN (near/ exact) -> 35
Out sizes        -> 603/139/284

=== Dedup @ /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID → dedup_near (near) ===
Train/Val/Test in -> 2004/573/87
Removed vs TEST  -> 59 from TRAIN/VAL
Removed VAL vs TRAIN (near/ exact) -> 170
Out sizes        -> 1962/386/87


'/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/dedup_near'

In [25]:
# EfficientNet-B0 on LOSO Coswera — using dedup_near splits
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T
from torchvision import models
from torch import nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

FOLD = "Cough_Audio_Coswera"
DED  = f"/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/{FOLD}/dedup_near"
train_csv, val_csv, test_csv = [f"{DED}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv])

LABEL2ID = {"healthy":0, "cancer":1}

# transforms (same harmonisation as before)
tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class DS(Dataset):
    def __init__(self, csv):
        df = pd.read_csv(csv)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return tf(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = DS(train_csv), DS(val_csv), DS(test_csv)

# balanced sampler on TRAIN
ytr = train_ds.df["label"].map(LABEL2ID).values
cnts = np.bincount(ytr, minlength=2).astype(np.float32)
w = (cnts.sum() / (2*np.maximum(cnts,1))).astype(np.float32)
sampler = WeightedRandomSampler(w[ytr].tolist(), num_samples=len(ytr), replacement=True)

BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

# losses / opt
class_w = torch.tensor([cnts.sum()/(2*max(1,cnts[0])),
                        cnts.sum()/(2*max(1,cnts[1]))],
                       dtype=torch.float32, device=device)
crit = nn.CrossEntropyLoss(weight=class_w)
opt  = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train)
    losses, ys, ps = [], [], []
    for xb,yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            lg = model(xb); loss = crit(lg, yb)
            if train:
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        probs = torch.softmax(lg,1)[:,1].detach().cpu().numpy()
        ys.extend(yb.detach().cpu().numpy().tolist())
        ps.extend(probs.tolist())
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps >= 0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    pr, rc, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(ys, ps)
    except: auc = float('nan')
    return float(np.mean(losses)), acc, pr, rc, f1, auc

EPOCHS, PATIENCE = 20, 4
best_auc, best_state, best_ep, no_imp = -1.0, None, -1, 0
for ep in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, True)
    vl = run_epoch(val_loader, False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1] > best_auc:
        best_auc, best_ep = vl[-1], ep
        best_state = {k: v.detach().cpu() for k,v in model.state_dict().items()}
        no_imp = 0
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f"Early stop at {ep}, best val AUC {best_auc:.3f} @ epoch {best_ep}")
            break

# pick τ* on VAL (precision≥0.98 else best-F1), then evaluate TEST
if best_state is not None:
    model.load_state_dict(best_state, strict=False)
def choose_tau(y, p, target=0.98):
    ps = np.unique(p); mids = [(ps[i]+ps[i+1])/2 for i in range(len(ps)-1)]
    cands = np.array([0.0] + mids + [1.0])
    best = None
    for t in cands:
        yhat = (p>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if pr>=target:
            cand = (rc, f1, t, pr)
            if best is None or cand > best: best = cand
    if best: rc, f1, t, pr = best; return float(t), ("prec>=0.98", pr, rc, f1)
    # fallback best-F1
    bestF = (-1,None)
    for t in cands:
        yhat=(p>=t).astype(int); pr,rc,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if f1>bestF[0]: bestF=(f1,(t,pr,rc))
    f1,(t,pr,rc)=bestF; return float(t), ("bestF1", pr, rc, f1)

def collect(loader, ds):
    probs, ys = [], []
    with torch.no_grad():
        for xb,yb in loader:
            pr = torch.softmax(model(xb.to(device)),1)[:,1].cpu().numpy()
            probs += pr.tolist(); ys += yb.numpy().tolist()
    df = ds.df.copy(); df["y_true"]=ys; df["prob_cancer"]=probs; return df

vd, td = collect(val_loader, val_ds), collect(test_loader, test_ds)
tau, (policy, p_v, r_v, f_v) = choose_tau(vd["y_true"].values, vd["prob_cancer"].values, 0.98)

yt, pt = td["y_true"].values, td["prob_cancer"].values
yhat = (pt>=tau).astype(int)
# After you have yt, pt, yhat, tau, etc.
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

acc = accuracy_score(yt, yhat)  # <- compute accuracy separately
pr_t, rc_t, f1_t, _ = precision_recall_fscore_support(yt, yhat, average='binary', zero_division=0)
auc = roc_auc_score(yt, pt)
cm  = confusion_matrix(yt, yhat, labels=[0,1])

print(f"\nVAL τ*={tau:.6f} policy={policy} | VAL prec={p_v:.3f} rec={r_v:.3f} f1={f_v:.3f}")
print(f"[TEST @ τ*] acc={acc:.3f} prec={pr_t:.3f} rec={rc_t:.3f} f1={f1_t:.3f} auc={auc:.3f}")
print('CM:\n', cm)



Epoch 01 | train loss 0.2876 f1 0.915 auc 0.977 | val loss 0.2619 f1 0.500 auc 0.990
Epoch 02 | train loss 0.0216 f1 0.984 auc 0.998 | val loss 0.1100 f1 0.714 auc 0.996
Epoch 03 | train loss 0.0065 f1 0.991 auc 1.000 | val loss 0.0718 f1 0.769 auc 1.000
Epoch 04 | train loss 0.0031 f1 1.000 auc 1.000 | val loss 0.0562 f1 1.000 auc 1.000
Epoch 05 | train loss 0.0044 f1 0.993 auc 1.000 | val loss 0.0364 f1 1.000 auc 1.000
Epoch 06 | train loss 0.0025 f1 0.998 auc 1.000 | val loss 0.0331 f1 1.000 auc 1.000
Epoch 07 | train loss 0.0019 f1 1.000 auc 1.000 | val loss 0.0290 f1 1.000 auc 1.000
Early stop at 7, best val AUC 1.000 @ epoch 3

VAL τ*=0.781053 policy=prec>=0.98 | VAL prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM:
 [[281   0]
 [  0   3]]


In [26]:
# COUGHVID-held-out: near-dedup vs TEST + (optional) VAL vs TRAIN
from pathlib import Path
split_dir = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID"

# Reuse the function you already defined earlier:
_ = build_dedup_splits(split_dir,
                       out_subdir="dedup_near",
                       mode="near",
                       ah_t=5, dh_t=5,
                       dedup_val_against_train=True)



=== Dedup @ /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID → dedup_near (near) ===
Train/Val/Test in -> 2004/573/87
Removed vs TEST  -> 59 from TRAIN/VAL
Removed VAL vs TRAIN (near/ exact) -> 170
Out sizes        -> 1962/386/87


In [27]:
# EfficientNet-B0 on COUGHVID-held-out dedup_near
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T
from torchvision import models
from torch import nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

FOLD = "Cough_Audio_COUGHVID"
DED  = f"/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/{FOLD}/dedup_near"
train_csv, val_csv, test_csv = [f"{DED}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv])

LABEL2ID = {"healthy":0, "cancer":1}
tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class DS(Dataset):
    def __init__(self, csv):
        df = pd.read_csv(csv)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return tf(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = DS(train_csv), DS(val_csv), DS(test_csv)

# balanced sampler
ytr = train_ds.df["label"].map(LABEL2ID).values
cnts = np.bincount(ytr, minlength=2).astype(np.float32)
w = (cnts.sum() / (2*np.maximum(cnts,1))).astype(np.float32)
sampler = WeightedRandomSampler(w[ytr].tolist(), num_samples=len(ytr), replacement=True)

BATCH=32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

class_w = torch.tensor([cnts.sum()/(2*max(1,cnts[0])), cnts.sum()/(2*max(1,cnts[1]))],
                       dtype=torch.float32, device=device)
crit = nn.CrossEntropyLoss(weight=class_w)
opt  = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched= torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train); losses=[]; ys=[]; ps=[]
    for xb,yb in loader:
        xb,yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            lg = model(xb); loss = crit(lg,yb)
            if train:
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        pr = torch.softmax(lg,1)[:,1].detach().cpu().numpy()
        ys += yb.cpu().numpy().tolist(); ps += pr.tolist()
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps>=0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    prc, rec, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(ys, ps)
    except: auc = float('nan')
    return float(np.mean(losses)), acc, prc, rec, f1, auc

best_auc=-1; best_state=None; patience=4
for ep in range(1,21):
    tr = run_epoch(train_loader, True)
    vl = run_epoch(val_loader, False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1] > best_auc:
        best_auc = vl[-1]; best_state = {k: v.detach().cpu() for k,v in model.state_dict().items()}
        patience = 4
    else:
        patience -= 1
        if patience <= 0:
            print(f"Early stop at {ep}, best val AUC {best_auc:.3f}")
            break

if best_state is not None:
    model.load_state_dict(best_state, strict=False)

# choose τ* on VAL (precision≥0.98 else best-F1), then TEST
def choose_tau(y,p,target=0.98):
    ps = np.unique(p); mids=[(ps[i]+ps[i+1])/2 for i in range(len(ps)-1)]
    cands = np.array([0.0]+mids+[1.0]); pick=None
    for t in cands:
        yhat=(p>=t).astype(int)
        prc,rec,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if prc>=target:
            item=(rec,f1,t,prc)
            if pick is None or item>pick: pick=item
    if pick: rec,f1,t,prc = pick; return float(t), ("prec>=0.98", prc, rec, f1)
    # fallback best-F1
    best=(-1,None)
    for t in cands:
        yhat=(p>=t).astype(int)
        prc,rec,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if f1>best[0]: best=(f1,(t,prc,rec))
    f1,(t,prc,rec)=best; return float(t), ("bestF1", prc, rec, f1)

def collect(loader, ds):
    probs=[]; ys=[]
    with torch.no_grad():
        for xb,yb in loader:
            pr=torch.softmax(model(xb.to(device)),1)[:,1].cpu().numpy()
            probs+=pr.tolist(); ys+=yb.numpy().tolist()
    df=ds.df.copy(); df["y_true"]=ys; df["prob_cancer"]=probs; return df

vd, td = collect(val_loader, val_ds), collect(test_loader, test_ds)
tau, (policy, p_v, r_v, f_v) = choose_tau(vd["y_true"].values, vd["prob_cancer"].values, 0.98)

yt, pt = td["y_true"].values, td["prob_cancer"].values
yhat = (pt>=tau).astype(int)
acc = accuracy_score(yt, yhat)
pr_t, rc_t, f1_t, _ = precision_recall_fscore_support(yt, yhat, average='binary', zero_division=0)
auc = roc_auc_score(yt, pt)
cm  = confusion_matrix(yt, yhat, labels=[0,1])

print(f"\nVAL τ*={tau:.6f} policy={policy} | VAL prec={p_v:.3f} rec={r_v:.3f} f1={f_v:.3f}")
print(f"[TEST @ τ*] acc={acc:.3f} prec={pr_t:.3f} rec={rc_t:.3f} f1={f1_t:.3f} auc={auc:.3f}")
print('CM:\n', cm)


Epoch 01 | train loss 0.0844 f1 0.971 auc 0.997 | val loss 0.0926 f1 0.714 auc 0.998
Epoch 02 | train loss 0.0019 f1 0.997 auc 1.000 | val loss 0.0520 f1 0.714 auc 1.000
Epoch 03 | train loss 0.0014 f1 0.998 auc 1.000 | val loss 0.0351 f1 0.714 auc 1.000
Epoch 04 | train loss 0.0009 f1 0.997 auc 1.000 | val loss 0.0261 f1 0.833 auc 1.000
Epoch 05 | train loss 0.0007 f1 0.998 auc 1.000 | val loss 0.0203 f1 1.000 auc 1.000
Epoch 06 | train loss 0.0005 f1 0.999 auc 1.000 | val loss 0.0197 f1 1.000 auc 1.000
Early stop at 6, best val AUC 1.000

VAL τ*=0.972682 policy=prec>=0.98 | VAL prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM:
 [[84  0]
 [ 0  3]]


In [30]:
# EfficientNet-B0 with style-robust augmentation on LOSO Coswera (dedup_near)
import os, numpy as np, pandas as pd, torch, random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T
from torchvision import models
from torch import nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

random.seed(0); np.random.seed(0); torch.manual_seed(0)

FOLD = "Cough_Audio_Coswera"
DED  = f"/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/{FOLD}/dedup_near"
train_csv, val_csv, test_csv = [f"{DED}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv])

LABEL2ID = {"healthy":0, "cancer":1}

# --- helpers: per-image standardisation + SpecAug masks on tensors ---
def per_image_standardize(t):
    m = t.mean(); s = t.std()
    s = s if float(s) > 1e-6 else torch.tensor(1e-6, dtype=t.dtype, device=t.device)
    return (t - m) / s

def spec_mask(t, num_time=2, num_freq=2, max_time_frac=0.2, max_freq_frac=0.2):
    # t: (C,H,W) with C=1 or 3; assume H=freq, W=time
    C, H, W = t.shape
    out = t.clone()
    # time masks (vertical bars)
    for _ in range(num_time):
        w = max(1, int(random.uniform(0.05, max_time_frac) * W))
        x0 = random.randint(0, max(0, W - w))
        out[:, :, x0:x0+w] = 0.0
    # freq masks (horizontal bars)
    for _ in range(num_freq):
        h = max(1, int(random.uniform(0.05, max_freq_frac) * H))
        y0 = random.randint(0, max(0, H - h))
        out[:, y0:y0+h, :] = 0.0
    return out

from torch import nn  # you already have this above

class To3(nn.Module):
    def forward(self, t):  # t: (1,H,W)
        return t.repeat(3, 1, 1)

class SpecAug(nn.Module):
    def __init__(self, p=0.8, num_time=2, num_freq=2, max_time_frac=0.2, max_freq_frac=0.2):
        super().__init__()
        self.p = p
        self.kw = dict(num_time=num_time, num_freq=num_freq,
                       max_time_frac=max_time_frac, max_freq_frac=max_freq_frac)

    def forward(self, t):
        if random.random() < self.p:
            return spec_mask(t, **self.kw)
        return t


# --- transforms ---
train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),               # (1,H,W) in [0,1]
    T.Lambda(per_image_standardize),
    To3(),                      # -> (3,H,W)
    T.RandomApply([T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))], p=0.7),
    SpecAug(p=0.8, num_time=2, num_freq=2, max_time_frac=0.2, max_freq_frac=0.2),
    T.RandomErasing(p=0.5, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0.0),
])

eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(per_image_standardize),
    To3(),
])

class DS(Dataset):
    def __init__(self, csv, tfm):
        df = pd.read_csv(csv)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df; self.tfm = tfm
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = DS(train_csv, train_tf), DS(val_csv, eval_tf), DS(test_csv, eval_tf)

# balanced sampler
ytr = train_ds.df["label"].map(LABEL2ID).values
cnts = np.bincount(ytr, minlength=2).astype(np.float32)
w = (cnts.sum() / (2*np.maximum(cnts,1))).astype(np.float32)
sampler = WeightedRandomSampler(w[ytr].tolist(), num_samples=len(ytr), replacement=True)

BATCH=32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

class_w = torch.tensor([cnts.sum()/(2*max(1,cnts[0])), cnts.sum()/(2*max(1,cnts[1]))],
                       dtype=torch.float32, device=device)
crit = nn.CrossEntropyLoss(weight=class_w)
opt  = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched= torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train); losses=[]; ys=[]; ps=[]
    for xb,yb in loader:
        xb,yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            lg = model(xb); loss = crit(lg,yb)
            if train:
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        pr = torch.softmax(lg,1)[:,1].detach().cpu().numpy()
        ys += yb.cpu().numpy().tolist(); ps += pr.tolist()
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps>=0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    prc, rec, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(ys, ps)
    except: auc = float('nan')
    return float(np.mean(losses)), acc, prc, rec, f1, auc

best_auc=-1; best_state=None; patience=4
for ep in range(1,21):
    tr = run_epoch(train_loader, True)
    vl = run_epoch(val_loader, False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1] > best_auc:
        best_auc = vl[-1]; best_state = {k: v.detach().cpu() for k,v in model.state_dict().items()}
        patience = 4
    else:
        patience -= 1
        if patience <= 0:
            print(f"Early stop at {ep}, best val AUC {best_auc:.3f}")
            break
if best_state is not None:
    model.load_state_dict(best_state, strict=False)

# choose τ* on VAL (precision≥0.98 else best-F1), then TEST
def choose_tau(y,p,target=0.98):
    ps = np.unique(p); mids=[(ps[i]+ps[i+1])/2 for i in range(len(ps)-1)]
    cands = np.array([0.0]+mids+[1.0]); pick=None
    for t in cands:
        yhat=(p>=t).astype(int)
        prc,rec,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if prc>=target:
            item=(rec,f1,t,prc)
            if pick is None or item>pick: pick=item
    if pick: rec,f1,t,prc=pick; return float(t), ("prec>=0.98", prc, rec, f1)
    # fallback best-F1
    best=(-1,None)
    for t in cands:
        yhat=(p>=t).astype(int); prc,rec,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if f1>best[0]: best=(f1,(t,prc,rec))
    f1,(t,prc,rec)=best; return float(t), ("bestF1", prc, rec, f1)

def collect(loader, ds):
    probs=[]; ys=[]
    with torch.no_grad():
        for xb,yb in loader:
            pr=torch.softmax(model(xb.to(device)),1)[:,1].cpu().numpy()
            probs+=pr.tolist(); ys+=yb.numpy().tolist()
    df=ds.df.copy(); df["y_true"]=ys; df["prob_cancer"]=probs; return df

vd, td = collect(val_loader, val_ds), collect(test_loader, test_ds)
tau, (policy, p_v, r_v, f_v) = choose_tau(vd["y_true"].values, vd["prob_cancer"].values, 0.98)

yt, pt = td["y_true"].values, td["prob_cancer"].values
yhat = (pt>=tau).astype(int)
acc = accuracy_score(yt, yhat)
pr_t, rc_t, f1_t, _ = precision_recall_fscore_support(yt, yhat, average='binary', zero_division=0)
auc = roc_auc_score(yt, pt)
cm  = confusion_matrix(yt, yhat, labels=[0,1])

print(f"\nVAL τ*={tau:.6f} policy={policy} | VAL prec={p_v:.3f} rec={r_v:.3f} f1={f_v:.3f}")
print(f"[TEST @ τ*] acc={acc:.3f} prec={pr_t:.3f} rec={rc_t:.3f} f1={f1_t:.3f} auc={auc:.3f}")
print('CM:\n', cm)


Epoch 01 | train loss 0.3243 f1 0.835 auc 0.942 | val loss 0.3357 f1 0.233 auc 0.988
Epoch 02 | train loss 0.0386 f1 0.969 auc 0.999 | val loss 0.1780 f1 0.714 auc 0.994
Epoch 03 | train loss 0.0156 f1 0.982 auc 0.999 | val loss 0.1179 f1 0.714 auc 1.000
Epoch 04 | train loss 0.0087 f1 0.983 auc 1.000 | val loss 0.0816 f1 0.714 auc 1.000
Epoch 05 | train loss 0.0099 f1 0.983 auc 0.998 | val loss 0.0613 f1 0.769 auc 1.000
Epoch 06 | train loss 0.0080 f1 0.987 auc 1.000 | val loss 0.0450 f1 1.000 auc 1.000
Epoch 07 | train loss 0.0075 f1 0.995 auc 1.000 | val loss 0.0378 f1 1.000 auc 1.000
Early stop at 7, best val AUC 1.000

VAL τ*=0.953159 policy=prec>=0.98 | VAL prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM:
 [[281   0]
 [  0   3]]


In [31]:
# stricter near-duplicate removal (ahash/dhash ≤ 10) for Coswera fold
build_dedup_splits(
    "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera",
    out_subdir="dedup_strict",
    mode="near",
    ah_t=10, dh_t=10,
    dedup_val_against_train=True
)



=== Dedup @ /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera → dedup_strict (near) ===
Train/Val/Test in -> 624/178/284
Removed vs TEST  -> 198 from TRAIN/VAL
Removed VAL vs TRAIN (near/ exact) -> 55
Out sizes        -> 464/85/284


'/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/dedup_strict'

In [32]:
# EfficientNet-B0 with style-robust augmentation on LOSO Coswera (dedup_near)
import os, numpy as np, pandas as pd, torch, random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T
from torchvision import models
from torch import nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

random.seed(0); np.random.seed(0); torch.manual_seed(0)

FOLD = "Cough_Audio_Coswera"
DED  = f"/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/{FOLD}/dedup_strict"
# ... keep the rest unchanged (same augmentation + training code)

train_csv, val_csv, test_csv = [f"{DED}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv])

LABEL2ID = {"healthy":0, "cancer":1}

# --- helpers: per-image standardisation + SpecAug masks on tensors ---
def per_image_standardize(t):
    m = t.mean(); s = t.std()
    s = s if float(s) > 1e-6 else torch.tensor(1e-6, dtype=t.dtype, device=t.device)
    return (t - m) / s

def spec_mask(t, num_time=2, num_freq=2, max_time_frac=0.2, max_freq_frac=0.2):
    # t: (C,H,W) with C=1 or 3; assume H=freq, W=time
    C, H, W = t.shape
    out = t.clone()
    # time masks (vertical bars)
    for _ in range(num_time):
        w = max(1, int(random.uniform(0.05, max_time_frac) * W))
        x0 = random.randint(0, max(0, W - w))
        out[:, :, x0:x0+w] = 0.0
    # freq masks (horizontal bars)
    for _ in range(num_freq):
        h = max(1, int(random.uniform(0.05, max_freq_frac) * H))
        y0 = random.randint(0, max(0, H - h))
        out[:, y0:y0+h, :] = 0.0
    return out

from torch import nn  # you already have this above

class To3(nn.Module):
    def forward(self, t):  # t: (1,H,W)
        return t.repeat(3, 1, 1)

class SpecAug(nn.Module):
    def __init__(self, p=0.8, num_time=2, num_freq=2, max_time_frac=0.2, max_freq_frac=0.2):
        super().__init__()
        self.p = p
        self.kw = dict(num_time=num_time, num_freq=num_freq,
                       max_time_frac=max_time_frac, max_freq_frac=max_freq_frac)

    def forward(self, t):
        if random.random() < self.p:
            return spec_mask(t, **self.kw)
        return t


# --- transforms ---
train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),               # (1,H,W) in [0,1]
    T.Lambda(per_image_standardize),
    To3(),                      # -> (3,H,W)
    T.RandomApply([T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))], p=0.7),
    SpecAug(p=0.8, num_time=2, num_freq=2, max_time_frac=0.2, max_freq_frac=0.2),
    T.RandomErasing(p=0.5, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0.0),
])

eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(per_image_standardize),
    To3(),
])

class DS(Dataset):
    def __init__(self, csv, tfm):
        df = pd.read_csv(csv)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df; self.tfm = tfm
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = DS(train_csv, train_tf), DS(val_csv, eval_tf), DS(test_csv, eval_tf)

# balanced sampler
ytr = train_ds.df["label"].map(LABEL2ID).values
cnts = np.bincount(ytr, minlength=2).astype(np.float32)
w = (cnts.sum() / (2*np.maximum(cnts,1))).astype(np.float32)
sampler = WeightedRandomSampler(w[ytr].tolist(), num_samples=len(ytr), replacement=True)

BATCH=32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

class_w = torch.tensor([cnts.sum()/(2*max(1,cnts[0])), cnts.sum()/(2*max(1,cnts[1]))],
                       dtype=torch.float32, device=device)
crit = nn.CrossEntropyLoss(weight=class_w)
opt  = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched= torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train); losses=[]; ys=[]; ps=[]
    for xb,yb in loader:
        xb,yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            lg = model(xb); loss = crit(lg,yb)
            if train:
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        pr = torch.softmax(lg,1)[:,1].detach().cpu().numpy()
        ys += yb.cpu().numpy().tolist(); ps += pr.tolist()
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps>=0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    prc, rec, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(ys, ps)
    except: auc = float('nan')
    return float(np.mean(losses)), acc, prc, rec, f1, auc

best_auc=-1; best_state=None; patience=4
for ep in range(1,21):
    tr = run_epoch(train_loader, True)
    vl = run_epoch(val_loader, False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")
    if vl[-1] > best_auc:
        best_auc = vl[-1]; best_state = {k: v.detach().cpu() for k,v in model.state_dict().items()}
        patience = 4
    else:
        patience -= 1
        if patience <= 0:
            print(f"Early stop at {ep}, best val AUC {best_auc:.3f}")
            break
if best_state is not None:
    model.load_state_dict(best_state, strict=False)

# choose τ* on VAL (precision≥0.98 else best-F1), then TEST
def choose_tau(y,p,target=0.98):
    ps = np.unique(p); mids=[(ps[i]+ps[i+1])/2 for i in range(len(ps)-1)]
    cands = np.array([0.0]+mids+[1.0]); pick=None
    for t in cands:
        yhat=(p>=t).astype(int)
        prc,rec,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if prc>=target:
            item=(rec,f1,t,prc)
            if pick is None or item>pick: pick=item
    if pick: rec,f1,t,prc=pick; return float(t), ("prec>=0.98", prc, rec, f1)
    # fallback best-F1
    best=(-1,None)
    for t in cands:
        yhat=(p>=t).astype(int); prc,rec,f1,_=precision_recall_fscore_support(y,yhat,average='binary',zero_division=0)
        if f1>best[0]: best=(f1,(t,prc,rec))
    f1,(t,prc,rec)=best; return float(t), ("bestF1", prc, rec, f1)

def collect(loader, ds):
    probs=[]; ys=[]
    with torch.no_grad():
        for xb,yb in loader:
            pr=torch.softmax(model(xb.to(device)),1)[:,1].cpu().numpy()
            probs+=pr.tolist(); ys+=yb.numpy().tolist()
    df=ds.df.copy(); df["y_true"]=ys; df["prob_cancer"]=probs; return df

vd, td = collect(val_loader, val_ds), collect(test_loader, test_ds)
tau, (policy, p_v, r_v, f_v) = choose_tau(vd["y_true"].values, vd["prob_cancer"].values, 0.98)

yt, pt = td["y_true"].values, td["prob_cancer"].values
yhat = (pt>=tau).astype(int)
acc = accuracy_score(yt, yhat)
pr_t, rc_t, f1_t, _ = precision_recall_fscore_support(yt, yhat, average='binary', zero_division=0)
auc = roc_auc_score(yt, pt)
cm  = confusion_matrix(yt, yhat, labels=[0,1])

print(f"\nVAL τ*={tau:.6f} policy={policy} | VAL prec={p_v:.3f} rec={r_v:.3f} f1={f_v:.3f}")
print(f"[TEST @ τ*] acc={acc:.3f} prec={pr_t:.3f} rec={rc_t:.3f} f1={f1_t:.3f} auc={auc:.3f}")
print('CM:\n', cm)


Epoch 01 | train loss 0.3936 f1 0.815 auc 0.924 | val loss 0.4756 f1 0.229 auc 0.991
Epoch 02 | train loss 0.0757 f1 0.964 auc 0.999 | val loss 0.2977 f1 0.615 auc 0.991
Epoch 03 | train loss 0.0223 f1 0.985 auc 0.996 | val loss 0.1778 f1 0.727 auc 0.991
Epoch 04 | train loss 0.0142 f1 0.984 auc 0.998 | val loss 0.1263 f1 0.727 auc 1.000
Epoch 05 | train loss 0.0133 f1 0.974 auc 1.000 | val loss 0.0979 f1 0.727 auc 1.000
Epoch 06 | train loss 0.0097 f1 0.992 auc 1.000 | val loss 0.0761 f1 0.800 auc 1.000
Epoch 07 | train loss 0.0129 f1 0.977 auc 1.000 | val loss 0.0721 f1 0.727 auc 1.000
Epoch 08 | train loss 0.0087 f1 0.992 auc 1.000 | val loss 0.0520 f1 1.000 auc 1.000
Early stop at 8, best val AUC 1.000

VAL τ*=0.949108 policy=prec>=0.98 | VAL prec=1.000 rec=1.000 f1=1.000
[TEST @ τ*] acc=1.000 prec=1.000 rec=1.000 f1=1.000 auc=1.000
CM:
 [[281   0]
 [  0   3]]


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'

In [37]:
# Source-only baseline: can "source" alone predict "cancer"?
import os, pandas as pd, numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

def run_source_only(split_dir, title=None):
    assert os.path.exists(split_dir), f"Not found: {split_dir}"
    def rd(name): 
        p = os.path.join(split_dir, name + ".csv")
        assert os.path.exists(p), p
        return pd.read_csv(p)

    train, val, test = rd("train"), rd("val"), rd("test")

    fit_df = pd.concat([train, val], ignore_index=True)
    X_fit  = fit_df[["source"]].astype(str).values
    y_fit  = (fit_df["label"].astype(str).str.lower() == "cancer").astype(int).values

    X_test = test[["source"]].astype(str).values
    y_test = (test["label"].astype(str).str.lower() == "cancer").astype(int).values

    # sklearn compatibility
    try:
        enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        enc = OneHotEncoder(sparse=False, handle_unknown="ignore")

    Xf = enc.fit_transform(X_fit)
    Xt = enc.transform(X_test)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(Xf, y_fit)

    prob = clf.predict_proba(Xt)[:, 1]
    yhat = (prob >= 0.5).astype(int)

    acc = accuracy_score(y_test, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, yhat, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y_test, prob)
    except:
        auc = float("nan")
    cm  = confusion_matrix(y_test, yhat, labels=[0,1])

    title = title or split_dir
    print(f"\n=== SOURCE-ONLY: {title} ===")
    print(f"acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}")
    print("Confusion matrix (rows=true [healthy,cancer], cols=pred):\n", cm)
    print("\nTest source distribution:\n", test.groupby(['source','label']).size())

# Coswera-held-out (dedup_strict)
run_source_only(
    "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/dedup_strict",
    title="LOSO: Coswera held-out (dedup_strict)"
)

# (Optional) COUGHVID-held-out (dedup_near)
# run_source_only(
#     "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/dedup_near",
#     title="LOSO: COUGHVID held-out (dedup_near)"
# )



=== SOURCE-ONLY: LOSO: Coswera held-out (dedup_strict) ===
acc=1.000  prec=1.000  rec=1.000  f1=1.000  auc=1.000
Confusion matrix (rows=true [healthy,cancer], cols=pred):
 [[281   0]
 [  0   3]]

Test source distribution:
 source               label  
Cough_Audio_Coswera  healthy    281
kaggle_cancer_raw    cancer       3
dtype: int64


In [38]:
import os, pandas as pd, numpy as np

def rebalance_with_kaggle_healthy(base_dir, out_dir, target_kh_in_test=6, seed=42):
    os.makedirs(out_dir, exist_ok=True)
    def rd(name): 
        p = os.path.join(base_dir, name + ".csv")
        assert os.path.exists(p), p
        return pd.read_csv(p)
    train, val, test = rd("train"), rd("val"), rd("test")

    def tag(df, split): 
        df = df.copy(); df["split"]=split; return df
    all_df = pd.concat([tag(train,"train"), tag(val,"val"), tag(test,"test")], ignore_index=True)

    is_kh = (all_df["source"].astype(str)=="kaggle_normal_raw") & (all_df["label"].astype(str).str.lower()=="healthy")
    kh_train = all_df[(all_df["split"].isin(["train","val"])) & is_kh]
    kh_test  = all_df[(all_df["split"]=="test") & is_kh]
    need = max(0, target_kh_in_test - len(kh_test))

    moved = pd.DataFrame(columns=all_df.columns)
    if need > 0 and len(kh_train) > 0:
        moved = kh_train.sample(n=min(need, len(kh_train)), random_state=seed)
        all_df.loc[moved.index, "split"] = "test"

    # Write out new splits
    tr_new = all_df[all_df["split"]=="train"].drop(columns=["split"])
    va_new = all_df[all_df["split"]=="val"].drop(columns=["split"])
    te_new = all_df[all_df["split"]=="test"].drop(columns=["split"])

    tr_new.to_csv(os.path.join(out_dir,"train.csv"), index=False)
    va_new.to_csv(os.path.join(out_dir,"val.csv"),   index=False)
    te_new.to_csv(os.path.join(out_dir,"test.csv"),  index=False)

    def show(df, name):
        print(f"{name} n={len(df)}")
        print("  by label:\n", df["label"].value_counts())
        print("  by source (head):\n", df["source"].value_counts().head(), "\n")
    print(f"\n=== Rebalanced splits @ {out_dir} (target Kaggle-healthy in TEST = {target_kh_in_test}) ===")
    show(tr_new,"TRAIN"); show(va_new,"VAL"); show(te_new,"TEST")
    print("Moved to TEST (kaggle healthy):", len(moved))
    if len(moved):
        print("  Examples moved:\n", moved[["filepath","source","label"]].head())

# 1) Coswera-held-out (use your stricter dedup splits as base)
COSWERA_BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_Coswera/dedup_strict"
COSWERA_OUT  = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO_balanced/Cough_Audio_Coswera/balanced_strict"
rebalance_with_kaggle_healthy(COSWERA_BASE, COSWERA_OUT, target_kh_in_test=6)

# 2) COUGHVID-held-out (use near-dedup base you made)
COUGHVID_BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO/Cough_Audio_COUGHVID/dedup_near"
COUGHVID_OUT  = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO_balanced/Cough_Audio_COUGHVID/balanced_near"
rebalance_with_kaggle_healthy(COUGHVID_BASE, COUGHVID_OUT, target_kh_in_test=6)



=== Rebalanced splits @ /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO_balanced/Cough_Audio_Coswera/balanced_strict (target Kaggle-healthy in TEST = 6) ===
TRAIN n=459
  by label:
 label
healthy    440
cancer      19
Name: count, dtype: int64
  by source (head):
 source
Cough_Audio_COUGHVID    428
kaggle_cancer_raw        19
kaggle_normal_raw        12
Name: count, dtype: int64 

VAL n=84
  by label:
 label
healthy    80
cancer      4
Name: count, dtype: int64
  by source (head):
 source
Cough_Audio_COUGHVID    78
kaggle_cancer_raw        4
kaggle_normal_raw        2
Name: count, dtype: int64 

TEST n=290
  by label:
 label
healthy    287
cancer       3
Name: count, dtype: int64
  by source (head):
 source
Cough_Audio_Coswera    281
kaggle_normal_raw        6
kaggle_cancer_raw        3
Name: count, dtype: int64 

Moved to TEST (kaggle healthy): 6
  Examples moved:
                                               filepath             source  \
24   /

In [39]:
# Source-only baseline on rebalanced Coswera-held-out split
import os, pandas as pd, numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

def run_source_only(split_dir, title=None):
    def rd(name): 
        p = os.path.join(split_dir, name + ".csv")
        assert os.path.exists(p), p
        return pd.read_csv(p)

    train, val, test = rd("train"), rd("val"), rd("test")

    fit_df = pd.concat([train, val], ignore_index=True)
    X_fit  = fit_df[["source"]].astype(str).values
    y_fit  = (fit_df["label"].astype(str).str.lower() == "cancer").astype(int).values

    X_test = test[["source"]].astype(str).values
    y_test = (test["label"].astype(str).str.lower() == "cancer").astype(int).values

    # sklearn compatibility (newer uses sparse_output, older uses sparse)
    try:
        enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        enc = OneHotEncoder(sparse=False, handle_unknown="ignore")

    Xf = enc.fit_transform(X_fit)
    Xt = enc.transform(X_test)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(Xf, y_fit)

    prob = clf.predict_proba(Xt)[:, 1]
    yhat = (prob >= 0.5).astype(int)

    acc = accuracy_score(y_test, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, yhat, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y_test, prob)
    except:
        auc = float("nan")
    cm  = confusion_matrix(y_test, yhat, labels=[0,1])

    title = title or split_dir
    print(f"\n=== SOURCE-ONLY: {title} ===")
    print(f"acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}")
    print("Confusion matrix (rows=true [healthy,cancer], cols=pred):\n", cm)
    print("\nTest source distribution:\n", test.groupby(['source','label']).size())

COSWERA_BAL = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/LOSO_balanced/Cough_Audio_Coswera/balanced_strict"
run_source_only(COSWERA_BAL, "LOSO_balanced: Coswera held-out (balanced_strict)")



=== SOURCE-ONLY: LOSO_balanced: Coswera held-out (balanced_strict) ===
acc=1.000  prec=1.000  rec=1.000  f1=1.000  auc=1.000
Confusion matrix (rows=true [healthy,cancer], cols=pred):
 [[287   0]
 [  0   3]]

Test source distribution:
 source               label  
Cough_Audio_Coswera  healthy    281
kaggle_cancer_raw    cancer       3
kaggle_normal_raw    healthy      6
dtype: int64


In [42]:
# Kaggle-only 5-fold stratified splits (with 90/10 train/val inside each fold)
import os, pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

MASTER = "//workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Model_training/metadata/master_metadata_realpaths.csv"
OUTDIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(MASTER)
df["label"]  = df["label"].astype(str).str.lower().str.strip()
df["source"] = df["source"].astype(str).str.strip()

# keep only Kaggle cancer/healthy
df = df[df["source"].isin(["kaggle_cancer_raw","kaggle_normal_raw"])].copy()
df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)

print("Kaggle-only counts:")
print(df["label"].value_counts(), "\nby source:\n", df["source"].value_counts())

X = df["filepath"].values
y = (df["label"]=="cancer").astype(int).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

summary_rows = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
    fold_dir = os.path.join(OUTDIR, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    train_full = df.iloc[tr_idx].copy()
    test_df    = df.iloc[te_idx].copy()

    # 90/10 split (stratified) inside training for validation
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
    y_trfull = (train_full["label"]=="cancer").astype(int).values
    tr_sub_idx, val_idx = next(sss.split(train_full, y_trfull))
    train_df = train_full.iloc[tr_sub_idx].copy()
    val_df   = train_full.iloc[val_idx].copy()

    # save CSVs
    for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
        d.to_csv(os.path.join(fold_dir, f"{name}.csv"), index=False)

    # small summary per fold
    def ccount(d): return d["label"].value_counts().to_dict()
    tr_c, va_c, te_c = ccount(train_df), ccount(val_df), ccount(test_df)
    print(f"\nFold {fold}: n train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}")
    print("  train:", tr_c, " | val:", va_c, " | test:", te_c)

    summary_rows.append({
        "fold": fold,
        "train_n": len(train_df), "val_n": len(val_df), "test_n": len(test_df),
        "train_cancer": tr_c.get("cancer",0), "train_healthy": tr_c.get("healthy",0),
        "val_cancer": va_c.get("cancer",0),   "val_healthy": va_c.get("healthy",0),
        "test_cancer": te_c.get("cancer",0),  "test_healthy": te_c.get("healthy",0),
    })

# write a compact summary table
sum_df = pd.DataFrame(summary_rows)
sum_csv = os.path.join(OUTDIR, "kaggle_cv_summary.csv")
sum_df.to_csv(sum_csv, index=False)
print("\nSaved summary ->", sum_csv)
print(sum_df)


Kaggle-only counts:
label
cancer     30
healthy    24
Name: count, dtype: int64 
by source:
 source
kaggle_cancer_raw    30
kaggle_normal_raw    24
Name: count, dtype: int64

Fold 1: n train/val/test = 38/5/11
  train: {'cancer': 21, 'healthy': 17}  | val: {'cancer': 3, 'healthy': 2}  | test: {'cancer': 6, 'healthy': 5}

Fold 2: n train/val/test = 38/5/11
  train: {'cancer': 21, 'healthy': 17}  | val: {'cancer': 3, 'healthy': 2}  | test: {'cancer': 6, 'healthy': 5}

Fold 3: n train/val/test = 38/5/11
  train: {'cancer': 21, 'healthy': 17}  | val: {'cancer': 3, 'healthy': 2}  | test: {'cancer': 6, 'healthy': 5}

Fold 4: n train/val/test = 38/5/11
  train: {'cancer': 21, 'healthy': 17}  | val: {'cancer': 3, 'healthy': 2}  | test: {'cancer': 6, 'healthy': 5}

Fold 5: n train/val/test = 39/5/10
  train: {'cancer': 21, 'healthy': 18}  | val: {'cancer': 3, 'healthy': 2}  | test: {'cancer': 6, 'healthy': 4}

Saved summary -> /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machi

In [43]:
# Kaggle-only fold_1 loaders (no training yet)
import os, pandas as pd, numpy as np, torch, random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image

random.seed(0); np.random.seed(0); torch.manual_seed(0)

FOLD_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_1"
train_csv = f"{FOLD_DIR}/train.csv"
val_csv   = f"{FOLD_DIR}/val.csv"
test_csv  = f"{FOLD_DIR}/test.csv"
assert all(os.path.exists(p) for p in [train_csv, val_csv, test_csv]), "fold_1 CSVs missing"

LABEL2ID = {"healthy":0, "cancer":1}

def per_image_standardize(t):
    m, s = t.mean(), t.std()
    if float(s) < 1e-6: s = torch.tensor(1e-6, dtype=t.dtype, device=t.device)
    return (t - m) / s

class To3(torch.nn.Module):
    def forward(self, t):  # (1,H,W)
        return t.repeat(3,1,1)

train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(per_image_standardize),
    To3(),
    T.RandomApply([T.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05))], p=0.6),
    T.RandomErasing(p=0.3, scale=(0.02,0.08), ratio=(0.3,3.3), value=0.0),
])

eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(per_image_standardize),
    To3(),
])

class ImgDS(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df, self.tfm = df, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = ImgDS(train_csv, train_tf), ImgDS(val_csv, eval_tf), ImgDS(test_csv, eval_tf)

# class-balanced sampler (tiny data)
ytr = train_ds.df["label"].map(LABEL2ID).values
counts = np.bincount(ytr, minlength=2).astype(np.float32)
class_w = (counts.sum() / (2*np.maximum(counts,1))).astype(np.float32)
sample_w = class_w[ytr]
sampler = WeightedRandomSampler(sample_w.tolist(), num_samples=len(sample_w), replacement=True)

BATCH=8
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

# quick peek
xb, yb = next(iter(train_loader))
print("Train batch:", tuple(xb.shape), "| labels:", yb.tolist())
print("Train counts [healthy,cancer]:", counts.tolist())
print("Sampler class weights:", class_w.tolist())
print("Sizes -> train/val/test:", len(train_ds), len(val_ds), len(test_ds))


Train batch: (8, 3, 224, 224) | labels: [0, 1, 0, 1, 1, 0, 1, 1]
Train counts [healthy,cancer]: [17.0, 21.0]
Sampler class weights: [1.1176470518112183, 0.9047619104385376]
Sizes -> train/val/test: 38 5 11


In [44]:
# Train EfficientNet-B0 on Kaggle_CV/fold_1 and evaluate
import os, json, math, numpy as np, torch
from torch import nn
from torch.optim import AdamW
from torchvision import models
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    confusion_matrix, classification_report
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXP = os.path.join(FOLD_DIR, "effnet_fold1")
CKPT_DIR = os.path.join(EXP, "checkpoints")
PRED_DIR = os.path.join(EXP, "predictions")
REP_DIR  = os.path.join(EXP, "reports")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

# ----- model -----
effnet = models.efficientnet_b0(weights=None)  # keep None to avoid download issues
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
effnet.to(device)

# ----- loss with class weights (from your train counts) -----
counts = np.bincount(train_ds.df["label"].map({"healthy":0,"cancer":1}).values, minlength=2).astype(np.float32)
loss_w = (counts.sum() / (2*np.maximum(counts,1))).astype(np.float32)
crit = nn.CrossEntropyLoss(weight=torch.tensor(loss_w, dtype=torch.float32, device=device))

opt = AdamW(effnet.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    effnet.train(train)
    losses, Ys, Ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = effnet(xb)
            loss = crit(logits, yb)
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
        with torch.no_grad():
            prob = torch.softmax(logits, dim=1)[:,1]
            Ys.append(yb.detach().cpu().numpy())
            Ps.append(prob.detach().cpu().numpy())
    Y = np.concatenate(Ys) if len(Ys) else np.array([])
    P = np.concatenate(Ps) if len(Ps) else np.array([])
    yhat = (P >= 0.5).astype(int) if len(P) else np.array([])
    pr, rc, f1, _ = precision_recall_fscore_support(Y, yhat, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(Y, P)
    except:
        auc = float('nan')
    return np.mean(losses), f1, auc, Y, P

BEST, PATIENCE, epochs_no_improve = -np.inf, 6, 0
EPOCHS = 40
best_path = os.path.join(CKPT_DIR, "efficientnet_b0_fold1_best.pt")

for ep in range(1, EPOCHS+1):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, train=True)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, train=False)
    print(f"Epoch {ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST + 1e-6:
        BEST, epochs_no_improve = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        epochs_no_improve += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {ep} (best={BEST:.3f})"); break

# ----- load best & evaluate -----
effnet.load_state_dict(torch.load(best_path, map_location=device))
_, _, _, yv, pv = run_epoch(val_loader, train=False)
_, _, _, yt, pt = run_epoch(test_loader, train=False)

# choose operating threshold τ*
taus = np.linspace(0, 1, 1001)
best_f1, tau_f1 = -1, 0.5
for t in taus:
    yhat = (pv >= t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if f1 > best_f1:
        best_f1, tau_f1 = f1, float(t)

# high-precision policy (≥0.98) if achievable
tau_prec = None
for t in taus:
    yhat = (pv >= t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if pr >= 0.98:
        tau_prec = float(t); break

tau_star = tau_prec if tau_prec is not None else tau_f1
policy = "prec>=0.98" if tau_prec is not None else "maxF1"

# report on VAL
yhat_v = (pv >= tau_star).astype(int)
acc_v = accuracy_score(yv, yhat_v)
pr_v, rc_v, f1_v, _ = precision_recall_fscore_support(yv, yhat_v, average='binary', zero_division=0)
auc_v = roc_auc_score(yv, pv) if len(np.unique(yv))==2 else float('nan')
cm_v  = confusion_matrix(yv, yhat_v, labels=[0,1]).tolist()
print(f"\nVAL τ*={tau_star:.6f} policy={policy} -> acc={acc_v:.3f} prec={pr_v:.3f} rec={rc_v:.3f} f1={f1_v:.3f} auc={auc_v:.3f}")
print("VAL CM:\n", cm_v)

# report on TEST
yhat_t = (pt >= tau_star).astype(int)
acc_t = accuracy_score(yt, yhat_t)
pr_t, rc_t, f1_t, _ = precision_recall_fscore_support(yt, yhat_t, average='binary', zero_division=0)
auc_t = roc_auc_score(yt, pt) if len(np.unique(yt))==2 else float('nan')
cm_t  = confusion_matrix(yt, yhat_t, labels=[0,1]).tolist()
print(f"\nTEST @ τ* -> acc={acc_t:.3f} prec={pr_t:.3f} rec={rc_t:.3f} f1={f1_t:.3f} auc={auc_t:.3f}")
print("TEST CM:\n", cm_t)

# save predictions (val+test) for inspection
import pandas as pd
def pack(df, y_true, p, split):
    out = df[["filepath","label","source"]].copy()
    out["split"] = split
    out["y_true"] = y_true
    out["prob_cancer"] = p
    out["y_pred_tau*"] = (p >= tau_star).astype(int)
    return out

val_pred  = pack(val_ds.df,  yv, pv, "val")
test_pred = pack(test_ds.df, yt, pt, "test")
pred_all  = pd.concat([val_pred, test_pred], ignore_index=True)
pred_csv  = os.path.join(PRED_DIR, "effnet_fold1_val_test_predictions.csv")
pred_all.to_csv(pred_csv, index=False)

# save manifest + tau*
manifest = {
    "model": "EfficientNet-B0",
    "fold": 1,
    "paths": {"checkpoint": best_path, "predictions_csv": pred_csv},
    "tau_star": float(tau_star),
    "policy": policy,
    "val":  {"acc": float(acc_v), "prec": float(pr_v), "rec": float(rc_v), "f1": float(f1_v), "auc": float(auc_v), "cm": cm_v},
    "test": {"acc": float(acc_t), "prec": float(pr_t), "rec": float(rc_t), "f1": float(f1_t), "auc": float(auc_t), "cm": cm_t},
}
with open(os.path.join(EXP, "manifest_effnet_fold1.json"), "w") as f:
    json.dump(manifest, f, indent=2)

with open(os.path.join(REP_DIR, "operating_threshold.txt"), "w") as f:
    f.write(f"{tau_star:.6f}\n")

print("\n✅ Saved:")
print(" - checkpoint :", best_path)
print(" - predictions:", pred_csv)
print(" - manifest   :", os.path.join(EXP, 'manifest_effnet_fold1.json'))
print(" - τ* file    :", os.path.join(REP_DIR, 'operating_threshold.txt'))


Epoch 01 | train loss 0.6982 f1 0.389 auc 0.437 | val loss 0.6901 f1 0.750 auc 0.250
Epoch 02 | train loss 0.6543 f1 0.818 auc 0.750 | val loss 0.6885 f1 0.750 auc 0.167
Epoch 03 | train loss 0.6309 f1 0.681 auc 0.668 | val loss 0.6910 f1 0.750 auc 0.167
Epoch 04 | train loss 0.6119 f1 0.485 auc 0.771 | val loss 0.6946 f1 0.750 auc 0.667
Epoch 05 | train loss 0.5799 f1 0.698 auc 0.794 | val loss 0.6902 f1 0.750 auc 0.500
Epoch 06 | train loss 0.4908 f1 0.788 auc 0.917 | val loss 0.6885 f1 0.750 auc 0.417
Epoch 07 | train loss 0.5882 f1 0.571 auc 0.824 | val loss 0.6892 f1 0.750 auc 0.000
Epoch 08 | train loss 0.5233 f1 0.686 auc 0.839 | val loss 0.6894 f1 0.750 auc 0.000
Epoch 09 | train loss 0.3583 f1 0.882 auc 0.961 | val loss 0.6889 f1 0.750 auc 0.000
Epoch 10 | train loss 0.3649 f1 0.872 auc 0.936 | val loss 0.6885 f1 0.750 auc 0.000
Early stopping at epoch 10 (best=0.667)

VAL τ*=0.000000 policy=maxF1 -> acc=0.600 prec=0.600 rec=1.000 f1=0.750 auc=0.667
VAL CM:
 [[0, 2], [0, 3]]



In [45]:
# Rebuild Kaggle fold_1 loaders with ImageNet normalization (no sampler)
import os, pandas as pd, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

FOLD_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_1"
train_csv = f"{FOLD_DIR}/train.csv"
val_csv   = f"{FOLD_DIR}/val.csv"
test_csv  = f"{FOLD_DIR}/test.csv"
assert all(os.path.exists(p) for p in [train_csv, val_csv, test_csv])

LABEL2ID = {"healthy":0, "cancer":1}
IMAGENET_MEAN=(0.485,0.456,0.406)
IMAGENET_STD =(0.229,0.224,0.225)

train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),  # grayscale -> 3ch
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.20, scale=(0.02,0.08), ratio=(0.3,3.3), value=0.0),
])
eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDS(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df, self.tfm = df, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = ImgDS(train_csv, train_tf), ImgDS(val_csv, eval_tf), ImgDS(test_csv, eval_tf)

BATCH=8
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
counts = np.bincount(train_ds.df["label"].map(LABEL2ID).values, minlength=2)
print("Train batch:", tuple(xb.shape), "| labels:", yb.tolist())
print("Train counts [healthy,cancer]:", counts.tolist())
print("Sizes -> train/val/test:", len(train_ds), len(val_ds), len(test_ds))


Train batch: (8, 3, 224, 224) | labels: [1, 0, 0, 1, 0, 1, 1, 0]
Train counts [healthy,cancer]: [17, 21]
Sizes -> train/val/test: 38 5 11


In [46]:
import os, json, numpy as np, torch
from torch import nn
from torch.optim import AdamW
from torchvision import models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXP = os.path.join(FOLD_DIR, "effnet_fold1_imagenet")
CKPT_DIR = os.path.join(EXP, "checkpoints")
PRED_DIR = os.path.join(EXP, "predictions")
REP_DIR  = os.path.join(EXP, "reports")
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(REP_DIR, exist_ok=True)

# ----- model: ImageNet pretrained -----
try:
    effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
except Exception:
    # fallback if weights object name differs or offline
    effnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
effnet.to(device)

# class-weighted loss (no sampler)
counts = np.bincount(train_ds.df["label"].map({"healthy":0,"cancer":1}).values, minlength=2).astype(np.float32)
loss_w = (counts.sum() / (2*np.maximum(counts,1))).astype(np.float32)
crit = nn.CrossEntropyLoss(weight=torch.tensor(loss_w, dtype=torch.float32, device=device))

def run_epoch(loader, train=True, model=None, opt=None):
    model = model or effnet
    if train:
        model.train(True)
    else:
        model.train(False)
    losses, Ys, Ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = crit(logits, yb)
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
        with torch.no_grad():
            prob = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            Ys.append(yb.detach().cpu().numpy()); Ps.append(prob)
    Y = np.concatenate(Ys) if Ys else np.array([])
    P = np.concatenate(Ps) if Ps else np.array([])
    yhat = (P >= 0.5).astype(int) if len(P) else np.array([])
    pr, rc, f1, _ = precision_recall_fscore_support(Y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(Y, P)
    except: auc = float('nan')
    return float(np.mean(losses)), f1, auc, Y, P

# ---- Phase 1: freeze features, train classifier head only (warmup) ----
for p in effnet.features.parameters(): p.requires_grad = False
opt = AdamW(filter(lambda p: p.requires_grad, effnet.parameters()), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

BEST, PATIENCE, epochs_no_improve = -np.inf, 6, 0
best_path = os.path.join(CKPT_DIR, "efficientnet_b0_fold1_imagenet_best.pt")

for ep in range(1, 11):  # short warmup
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Warmup] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST + 1e-6:
        BEST, epochs_no_improve = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        epochs_no_improve += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if epochs_no_improve >= PATIENCE:
        break

# ---- Phase 2: unfreeze all, fine-tune with small LR ----
effnet.load_state_dict(torch.load(best_path, map_location=device))
for p in effnet.features.parameters(): p.requires_grad = True
opt = AdamW(effnet.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

BEST2, epochs_no_improve = -np.inf, 0
for ep in range(1, 31):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Finetune] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST2 + 1e-6:
        BEST2, epochs_no_improve = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        epochs_no_improve += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if epochs_no_improve >= PATIENCE:
        print(f"Early stop at finetune epoch {ep} (best={BEST2:.3f})"); break

# ----- Evaluate best & choose τ* on val -----
effnet.load_state_dict(torch.load(best_path, map_location=device))
_, _, _, yv, pv = run_epoch(val_loader, False, effnet)
_, _, _, yt, pt = run_epoch(test_loader, False, effnet)

taus = np.linspace(0,1,1001)
best_f1, tau_f1 = -1, 0.5
for t in taus:
    yhat = (pv>=t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if f1 > best_f1: best_f1, tau_f1 = f1, float(t)

tau_prec = None
for t in taus:
    yhat = (pv>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if pr >= 0.98:
        tau_prec = float(t); break

tau_star = tau_prec if tau_prec is not None else tau_f1
policy = "prec>=0.98" if tau_prec is not None else "maxF1"

from sklearn.metrics import classification_report
def report(y, p, t, tag):
    yhat = (p>=t).astype(int)
    acc = accuracy_score(y, yhat); pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(y, p)
    except: auc = float('nan')
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    print(f"\n[{tag} @ τ*={t:.6f}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f}")
    print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)
    return acc, pr, rc, f1, auc, cm

val_metrics  = report(yv, pv, tau_star, "VAL")
test_metrics = report(yt, pt, tau_star, "TEST")

# ----- Save artifacts -----
import pandas as pd
def pack(df, y_true, p, split):
    out = df[["filepath","label","source"]].copy()
    out["split"] = split; out["y_true"]=y_true; out["prob_cancer"]=p
    out["y_pred_tau*"] = (p>=tau_star).astype(int)
    return out

val_pred  = pack(val_ds.df,  yv, pv, "val")
test_pred = pack(test_ds.df, yt, pt, "test")
pred_all  = pd.concat([val_pred, test_pred], ignore_index=True)
pred_csv  = os.path.join(PRED_DIR, "effnet_fold1_imagenet_val_test_predictions.csv")
pred_all.to_csv(pred_csv, index=False)

manifest = {
    "model": "EfficientNet-B0 (ImageNet init)",
    "fold": 1, "policy": policy, "tau_star": float(tau_star),
    "paths": {"checkpoint": os.path.join(CKPT_DIR,"efficientnet_b0_fold1_imagenet_best.pt"),
              "predictions_csv": pred_csv,
              "tau_file": os.path.join(REP_DIR, "operating_threshold.txt")},
    "val":  {"acc": float(val_metrics[0]), "prec": float(val_metrics[1]), "rec": float(val_metrics[2]),
             "f1": float(val_metrics[3]), "auc": float(val_metrics[4]), "cm": val_metrics[5]},
    "test": {"acc": float(test_metrics[0]), "prec": float(test_metrics[1]), "rec": float(test_metrics[2]),
             "f1": float(test_metrics[3]), "auc": float(test_metrics[4]), "cm": test_metrics[5]},
}
with open(os.path.join(EXP, "manifest_effnet_fold1_imagenet.json"), "w") as f: json.dump(manifest, f, indent=2)
with open(os.path.join(REP_DIR, "operating_threshold.txt"), "w") as f: f.write(f"{tau_star:.6f}\n")

print("\n✅ Saved:")
print(" - checkpoint :", manifest["paths"]["checkpoint"])
print(" - predictions:", manifest["paths"]["predictions_csv"])
print(" - manifest   :", os.path.join(EXP, 'manifest_effnet_fold1_imagenet.json'))
print(" - τ* file    :", manifest["paths"]["tau_file"])


[Warmup] Ep01 | train loss 0.5582 f1 0.686 auc 0.871 | val loss 0.5847 f1 0.857 auc 1.000
[Warmup] Ep02 | train loss 0.3539 f1 0.927 auc 0.964 | val loss 0.6366 f1 0.750 auc 1.000
[Warmup] Ep03 | train loss 0.2327 f1 0.976 auc 1.000 | val loss 0.5321 f1 0.750 auc 1.000
[Warmup] Ep04 | train loss 0.1827 f1 0.977 auc 0.992 | val loss 0.4229 f1 0.857 auc 1.000
[Warmup] Ep05 | train loss 0.1983 f1 0.977 auc 1.000 | val loss 0.4206 f1 0.857 auc 1.000
[Warmup] Ep06 | train loss 0.2686 f1 0.930 auc 0.964 | val loss 0.4292 f1 0.857 auc 1.000
[Warmup] Ep07 | train loss 0.1598 f1 0.976 auc 0.997 | val loss 0.3074 f1 0.857 auc 1.000
[Finetune] Ep01 | train loss 0.3816 f1 0.952 auc 0.966 | val loss 0.5471 f1 0.857 auc 1.000
[Finetune] Ep02 | train loss 0.3026 f1 0.895 auc 1.000 | val loss 0.4434 f1 0.857 auc 1.000
[Finetune] Ep03 | train loss 0.2507 f1 0.952 auc 0.994 | val loss 0.4210 f1 1.000 auc 1.000
[Finetune] Ep04 | train loss 0.2255 f1 0.950 auc 0.997 | val loss 0.3754 f1 0.857 auc 1.000
[F

In [47]:
# EfficientNet-B0 (ImageNet) on Kaggle_CV/fold_2 — train, pick τ*, evaluate, save
import os, json, numpy as np, torch, pandas as pd
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import models, transforms as T
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# ---------- Paths ----------
FOLD = 2
BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
FOLD_DIR = f"{BASE}/fold_{FOLD}"
train_csv, val_csv, test_csv = [f"{FOLD_DIR}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv]), "Fold-2 CSVs missing."

EXP      = os.path.join(FOLD_DIR, "effnet_fold2_imagenet")
CKPT_DIR = os.path.join(EXP, "checkpoints")
PRED_DIR = os.path.join(EXP, "predictions")
REP_DIR  = os.path.join(EXP, "reports")
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(REP_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL2ID = {"healthy":0, "cancer":1}
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)

# ---------- Data ----------
train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.20, scale=(0.02,0.08), ratio=(0.3,3.3), value=0.0),
])
eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDS(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df, self.tfm = df, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = ImgDS(train_csv, train_tf), ImgDS(val_csv, eval_tf), ImgDS(test_csv, eval_tf)
BATCH=8
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

# ---------- Model ----------
try:
    effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
except Exception:
    effnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
effnet.to(device)

# class-weighted CE (small set)
counts = np.bincount(train_ds.df["label"].map(LABEL2ID).values, minlength=2).astype(np.float32)
loss_w = (counts.sum() / (2*np.maximum(counts,1))).astype(np.float32)
crit = nn.CrossEntropyLoss(weight=torch.tensor(loss_w, dtype=torch.float32, device=device))

def run_epoch(loader, train=True, model=None, opt=None):
    model = model or effnet
    model.train(train)
    losses, Ys, Ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = crit(logits, yb)
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
        with torch.no_grad():
            prob = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            Ys.append(yb.detach().cpu().numpy()); Ps.append(prob)
    Y = np.concatenate(Ys) if Ys else np.array([])
    P = np.concatenate(Ps) if Ps else np.array([])
    yhat = (P>=0.5).astype(int) if len(P) else np.array([])
    pr, rc, f1, _ = precision_recall_fscore_support(Y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(Y,P)
    except: auc = float('nan')
    return float(np.mean(losses)), f1, auc, Y, P

# ---------- Phase 1: warmup (freeze features) ----------
for p in effnet.features.parameters(): p.requires_grad=False
opt = AdamW(filter(lambda p: p.requires_grad, effnet.parameters()), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
BEST, PATIENCE, noimp = -np.inf, 6, 0
best_path = os.path.join(CKPT_DIR, "efficientnet_b0_fold2_imagenet_best.pt")

for ep in range(1, 11):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Warmup] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST + 1e-6:
        BEST, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE: break

# ---------- Phase 2: finetune (unfreeze) ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
for p in effnet.features.parameters(): p.requires_grad=True
opt = AdamW(effnet.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

BEST2, noimp = -np.inf, 0
for ep in range(1, 31):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Finetune] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST2 + 1e-6:
        BEST2, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE:
        print(f"Early stop at finetune epoch {ep} (best={BEST2:.3f})"); break

# ---------- Threshold selection ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
_, _, _, yv, pv = run_epoch(val_loader, False, effnet)
_, _, _, yt, pt = run_epoch(test_loader, False, effnet)

taus = np.linspace(0,1,1001)
best_f1, tau_f1 = -1, 0.5
for t in taus:
    yhat = (pv>=t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if f1 > best_f1: best_f1, tau_f1 = f1, float(t)

tau_prec = None
for t in taus:
    yhat = (pv>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if pr >= 0.98:
        tau_prec = float(t); break

tau_star = tau_prec if tau_prec is not None else tau_f1
policy = "prec>=0.98" if tau_prec is not None else "maxF1"

def report(y, p, t, tag):
    yhat = (p>=t).astype(int)
    acc = accuracy_score(y, yhat); pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(y, p)
    except: auc = float('nan')
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    print(f"\n[{tag} @ τ*={t:.6f}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f}")
    print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)
    return acc, pr, rc, f1, auc, cm

val_metrics  = report(yv, pv, tau_star, "VAL")
test_metrics = report(yt, pt, tau_star, "TEST")

# ---------- Save artifacts ----------
def pack(df, y_true, p, split):
    out = df[["filepath","label","source"]].copy()
    out["split"]=split; out["y_true"]=y_true; out["prob_cancer"]=p
    out["y_pred_tau*"]=(p>=tau_star).astype(int)
    return out

val_pred  = pack(val_ds.df,  yv, pv, "val")
test_pred = pack(test_ds.df, yt, pt, "test")
pred_all  = pd.concat([val_pred, test_pred], ignore_index=True)
pred_csv  = os.path.join(PRED_DIR, "effnet_fold2_imagenet_val_test_predictions.csv")
pred_all.to_csv(pred_csv, index=False)

manifest = {
    "model": "EfficientNet-B0 (ImageNet init)",
    "fold": FOLD, "policy": policy, "tau_star": float(tau_star),
    "paths": {"checkpoint": os.path.join(CKPT_DIR,"efficientnet_b0_fold2_imagenet_best.pt"),
              "predictions_csv": pred_csv,
              "tau_file": os.path.join(REP_DIR, "operating_threshold.txt")},
    "val":  {"acc": float(val_metrics[0]), "prec": float(val_metrics[1]), "rec": float(val_metrics[2]),
             "f1": float(val_metrics[3]), "auc": float(val_metrics[4]), "cm": val_metrics[5]},
    "test": {"acc": float(test_metrics[0]), "prec": float(test_metrics[1]), "rec": float(test_metrics[2]),
             "f1": float(test_metrics[3]), "auc": float(test_metrics[4]), "cm": test_metrics[5]},
}
with open(os.path.join(EXP, "manifest_effnet_fold2_imagenet.json"), "w") as f: json.dump(manifest, f, indent=2)
with open(os.path.join(REP_DIR, "operating_threshold.txt"), "w") as f: f.write(f"{tau_star:.6f}\n")

print("\n✅ Saved:")
print(" - checkpoint :", manifest["paths"]["checkpoint"])
print(" - predictions:", manifest["paths"]["predictions_csv"])
print(" - manifest   :", os.path.join(EXP, 'manifest_effnet_fold2_imagenet.json'))
print(" - τ* file    :", manifest["paths"]["tau_file"])


[Warmup] Ep01 | train loss 0.6278 f1 0.773 auc 0.697 | val loss 0.7389 f1 0.571 auc 0.167
[Warmup] Ep02 | train loss 0.4130 f1 0.905 auc 0.966 | val loss 0.7139 f1 0.750 auc 0.500
[Warmup] Ep03 | train loss 0.2514 f1 0.927 auc 0.992 | val loss 0.6913 f1 0.750 auc 0.833
[Warmup] Ep04 | train loss 0.2143 f1 0.927 auc 0.989 | val loss 0.6635 f1 0.750 auc 1.000
[Warmup] Ep05 | train loss 0.2766 f1 0.878 auc 0.966 | val loss 0.5092 f1 0.750 auc 1.000
[Warmup] Ep06 | train loss 0.1745 f1 0.976 auc 0.997 | val loss 0.4739 f1 0.750 auc 1.000
[Warmup] Ep07 | train loss 0.1103 f1 0.977 auc 0.997 | val loss 0.4341 f1 0.750 auc 1.000
[Warmup] Ep08 | train loss 0.2298 f1 0.930 auc 0.969 | val loss 0.3467 f1 0.857 auc 1.000
[Warmup] Ep09 | train loss 0.1589 f1 0.952 auc 0.992 | val loss 0.2860 f1 1.000 auc 1.000
[Warmup] Ep10 | train loss 0.0984 f1 0.976 auc 1.000 | val loss 0.1834 f1 1.000 auc 1.000
[Finetune] Ep01 | train loss 0.1907 f1 0.977 auc 0.986 | val loss 0.5311 f1 0.750 auc 1.000
[Finetun

In [48]:
# EfficientNet-B0 (ImageNet) on Kaggle_CV/fold_2 — train, pick τ*, evaluate, save
import os, json, numpy as np, torch, pandas as pd
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import models, transforms as T
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# ---------- Paths ----------
FOLD = 3
BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
FOLD_DIR = f"{BASE}/fold_{FOLD}"
train_csv, val_csv, test_csv = [f"{FOLD_DIR}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv]), "Fold-2 CSVs missing."

EXP      = os.path.join(FOLD_DIR, "effnet_fold2_imagenet")
CKPT_DIR = os.path.join(EXP, "checkpoints")
PRED_DIR = os.path.join(EXP, "predictions")
REP_DIR  = os.path.join(EXP, "reports")
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(REP_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL2ID = {"healthy":0, "cancer":1}
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)

# ---------- Data ----------
train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.20, scale=(0.02,0.08), ratio=(0.3,3.3), value=0.0),
])
eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDS(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df, self.tfm = df, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = ImgDS(train_csv, train_tf), ImgDS(val_csv, eval_tf), ImgDS(test_csv, eval_tf)
BATCH=8
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

# ---------- Model ----------
try:
    effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
except Exception:
    effnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
effnet.to(device)

# class-weighted CE (small set)
counts = np.bincount(train_ds.df["label"].map(LABEL2ID).values, minlength=2).astype(np.float32)
loss_w = (counts.sum() / (2*np.maximum(counts,1))).astype(np.float32)
crit = nn.CrossEntropyLoss(weight=torch.tensor(loss_w, dtype=torch.float32, device=device))

def run_epoch(loader, train=True, model=None, opt=None):
    model = model or effnet
    model.train(train)
    losses, Ys, Ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = crit(logits, yb)
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
        with torch.no_grad():
            prob = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            Ys.append(yb.detach().cpu().numpy()); Ps.append(prob)
    Y = np.concatenate(Ys) if Ys else np.array([])
    P = np.concatenate(Ps) if Ps else np.array([])
    yhat = (P>=0.5).astype(int) if len(P) else np.array([])
    pr, rc, f1, _ = precision_recall_fscore_support(Y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(Y,P)
    except: auc = float('nan')
    return float(np.mean(losses)), f1, auc, Y, P

# ---------- Phase 1: warmup (freeze features) ----------
for p in effnet.features.parameters(): p.requires_grad=False
opt = AdamW(filter(lambda p: p.requires_grad, effnet.parameters()), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
BEST, PATIENCE, noimp = -np.inf, 6, 0
best_path = os.path.join(CKPT_DIR, "efficientnet_b0_fold2_imagenet_best.pt")

for ep in range(1, 11):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Warmup] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST + 1e-6:
        BEST, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE: break

# ---------- Phase 2: finetune (unfreeze) ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
for p in effnet.features.parameters(): p.requires_grad=True
opt = AdamW(effnet.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

BEST2, noimp = -np.inf, 0
for ep in range(1, 31):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Finetune] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST2 + 1e-6:
        BEST2, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE:
        print(f"Early stop at finetune epoch {ep} (best={BEST2:.3f})"); break

# ---------- Threshold selection ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
_, _, _, yv, pv = run_epoch(val_loader, False, effnet)
_, _, _, yt, pt = run_epoch(test_loader, False, effnet)

taus = np.linspace(0,1,1001)
best_f1, tau_f1 = -1, 0.5
for t in taus:
    yhat = (pv>=t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if f1 > best_f1: best_f1, tau_f1 = f1, float(t)

tau_prec = None
for t in taus:
    yhat = (pv>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if pr >= 0.98:
        tau_prec = float(t); break

tau_star = tau_prec if tau_prec is not None else tau_f1
policy = "prec>=0.98" if tau_prec is not None else "maxF1"

def report(y, p, t, tag):
    yhat = (p>=t).astype(int)
    acc = accuracy_score(y, yhat); pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(y, p)
    except: auc = float('nan')
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    print(f"\n[{tag} @ τ*={t:.6f}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f}")
    print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)
    return acc, pr, rc, f1, auc, cm

val_metrics  = report(yv, pv, tau_star, "VAL")
test_metrics = report(yt, pt, tau_star, "TEST")

# ---------- Save artifacts ----------
def pack(df, y_true, p, split):
    out = df[["filepath","label","source"]].copy()
    out["split"]=split; out["y_true"]=y_true; out["prob_cancer"]=p
    out["y_pred_tau*"]=(p>=tau_star).astype(int)
    return out

val_pred  = pack(val_ds.df,  yv, pv, "val")
test_pred = pack(test_ds.df, yt, pt, "test")
pred_all  = pd.concat([val_pred, test_pred], ignore_index=True)
pred_csv  = os.path.join(PRED_DIR, "effnet_fold2_imagenet_val_test_predictions.csv")
pred_all.to_csv(pred_csv, index=False)

manifest = {
    "model": "EfficientNet-B0 (ImageNet init)",
    "fold": FOLD, "policy": policy, "tau_star": float(tau_star),
    "paths": {"checkpoint": os.path.join(CKPT_DIR,"efficientnet_b0_fold2_imagenet_best.pt"),
              "predictions_csv": pred_csv,
              "tau_file": os.path.join(REP_DIR, "operating_threshold.txt")},
    "val":  {"acc": float(val_metrics[0]), "prec": float(val_metrics[1]), "rec": float(val_metrics[2]),
             "f1": float(val_metrics[3]), "auc": float(val_metrics[4]), "cm": val_metrics[5]},
    "test": {"acc": float(test_metrics[0]), "prec": float(test_metrics[1]), "rec": float(test_metrics[2]),
             "f1": float(test_metrics[3]), "auc": float(test_metrics[4]), "cm": test_metrics[5]},
}
with open(os.path.join(EXP, "manifest_effnet_fold2_imagenet.json"), "w") as f: json.dump(manifest, f, indent=2)
with open(os.path.join(REP_DIR, "operating_threshold.txt"), "w") as f: f.write(f"{tau_star:.6f}\n")

print("\n✅ Saved:")
print(" - checkpoint :", manifest["paths"]["checkpoint"])
print(" - predictions:", manifest["paths"]["predictions_csv"])
print(" - manifest   :", os.path.join(EXP, 'manifest_effnet_fold2_imagenet.json'))
print(" - τ* file    :", manifest["paths"]["tau_file"])


[Warmup] Ep01 | train loss 0.7501 f1 0.653 auc 0.468 | val loss 0.7586 f1 0.571 auc 0.333
[Warmup] Ep02 | train loss 0.5064 f1 0.722 auc 0.874 | val loss 0.5735 f1 0.857 auc 0.833
[Warmup] Ep03 | train loss 0.3793 f1 0.811 auc 0.958 | val loss 0.4985 f1 0.857 auc 1.000
[Warmup] Ep04 | train loss 0.3260 f1 0.900 auc 0.955 | val loss 0.4812 f1 0.857 auc 1.000
[Warmup] Ep05 | train loss 0.2040 f1 0.930 auc 0.994 | val loss 0.4574 f1 0.750 auc 1.000
[Warmup] Ep06 | train loss 0.3702 f1 0.884 auc 0.908 | val loss 0.4691 f1 0.750 auc 1.000
[Warmup] Ep07 | train loss 0.1809 f1 0.977 auc 0.989 | val loss 0.4155 f1 0.750 auc 1.000
[Warmup] Ep08 | train loss 0.1463 f1 0.950 auc 1.000 | val loss 0.4039 f1 0.857 auc 1.000
[Warmup] Ep09 | train loss 0.1196 f1 1.000 auc 1.000 | val loss 0.3308 f1 1.000 auc 1.000
[Finetune] Ep01 | train loss 0.2652 f1 0.952 auc 0.992 | val loss 0.4261 f1 0.857 auc 1.000
[Finetune] Ep02 | train loss 0.2769 f1 0.923 auc 0.989 | val loss 0.4100 f1 0.857 auc 1.000
[Finet

In [49]:
# EfficientNet-B0 (ImageNet) on Kaggle_CV/fold_2 — train, pick τ*, evaluate, save
import os, json, numpy as np, torch, pandas as pd
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import models, transforms as T
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# ---------- Paths ----------
FOLD = 4
BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
FOLD_DIR = f"{BASE}/fold_{FOLD}"
train_csv, val_csv, test_csv = [f"{FOLD_DIR}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv]), "Fold-2 CSVs missing."

EXP      = os.path.join(FOLD_DIR, "effnet_fold2_imagenet")
CKPT_DIR = os.path.join(EXP, "checkpoints")
PRED_DIR = os.path.join(EXP, "predictions")
REP_DIR  = os.path.join(EXP, "reports")
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(REP_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL2ID = {"healthy":0, "cancer":1}
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)

# ---------- Data ----------
train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.20, scale=(0.02,0.08), ratio=(0.3,3.3), value=0.0),
])
eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDS(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df, self.tfm = df, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = ImgDS(train_csv, train_tf), ImgDS(val_csv, eval_tf), ImgDS(test_csv, eval_tf)
BATCH=8
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

# ---------- Model ----------
try:
    effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
except Exception:
    effnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
effnet.to(device)

# class-weighted CE (small set)
counts = np.bincount(train_ds.df["label"].map(LABEL2ID).values, minlength=2).astype(np.float32)
loss_w = (counts.sum() / (2*np.maximum(counts,1))).astype(np.float32)
crit = nn.CrossEntropyLoss(weight=torch.tensor(loss_w, dtype=torch.float32, device=device))

def run_epoch(loader, train=True, model=None, opt=None):
    model = model or effnet
    model.train(train)
    losses, Ys, Ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = crit(logits, yb)
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
        with torch.no_grad():
            prob = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            Ys.append(yb.detach().cpu().numpy()); Ps.append(prob)
    Y = np.concatenate(Ys) if Ys else np.array([])
    P = np.concatenate(Ps) if Ps else np.array([])
    yhat = (P>=0.5).astype(int) if len(P) else np.array([])
    pr, rc, f1, _ = precision_recall_fscore_support(Y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(Y,P)
    except: auc = float('nan')
    return float(np.mean(losses)), f1, auc, Y, P

# ---------- Phase 1: warmup (freeze features) ----------
for p in effnet.features.parameters(): p.requires_grad=False
opt = AdamW(filter(lambda p: p.requires_grad, effnet.parameters()), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
BEST, PATIENCE, noimp = -np.inf, 6, 0
best_path = os.path.join(CKPT_DIR, "efficientnet_b0_fold2_imagenet_best.pt")

for ep in range(1, 11):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Warmup] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST + 1e-6:
        BEST, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE: break

# ---------- Phase 2: finetune (unfreeze) ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
for p in effnet.features.parameters(): p.requires_grad=True
opt = AdamW(effnet.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

BEST2, noimp = -np.inf, 0
for ep in range(1, 31):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Finetune] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST2 + 1e-6:
        BEST2, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE:
        print(f"Early stop at finetune epoch {ep} (best={BEST2:.3f})"); break

# ---------- Threshold selection ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
_, _, _, yv, pv = run_epoch(val_loader, False, effnet)
_, _, _, yt, pt = run_epoch(test_loader, False, effnet)

taus = np.linspace(0,1,1001)
best_f1, tau_f1 = -1, 0.5
for t in taus:
    yhat = (pv>=t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if f1 > best_f1: best_f1, tau_f1 = f1, float(t)

tau_prec = None
for t in taus:
    yhat = (pv>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if pr >= 0.98:
        tau_prec = float(t); break

tau_star = tau_prec if tau_prec is not None else tau_f1
policy = "prec>=0.98" if tau_prec is not None else "maxF1"

def report(y, p, t, tag):
    yhat = (p>=t).astype(int)
    acc = accuracy_score(y, yhat); pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(y, p)
    except: auc = float('nan')
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    print(f"\n[{tag} @ τ*={t:.6f}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f}")
    print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)
    return acc, pr, rc, f1, auc, cm

val_metrics  = report(yv, pv, tau_star, "VAL")
test_metrics = report(yt, pt, tau_star, "TEST")

# ---------- Save artifacts ----------
def pack(df, y_true, p, split):
    out = df[["filepath","label","source"]].copy()
    out["split"]=split; out["y_true"]=y_true; out["prob_cancer"]=p
    out["y_pred_tau*"]=(p>=tau_star).astype(int)
    return out

val_pred  = pack(val_ds.df,  yv, pv, "val")
test_pred = pack(test_ds.df, yt, pt, "test")
pred_all  = pd.concat([val_pred, test_pred], ignore_index=True)
pred_csv  = os.path.join(PRED_DIR, "effnet_fold2_imagenet_val_test_predictions.csv")
pred_all.to_csv(pred_csv, index=False)

manifest = {
    "model": "EfficientNet-B0 (ImageNet init)",
    "fold": FOLD, "policy": policy, "tau_star": float(tau_star),
    "paths": {"checkpoint": os.path.join(CKPT_DIR,"efficientnet_b0_fold2_imagenet_best.pt"),
              "predictions_csv": pred_csv,
              "tau_file": os.path.join(REP_DIR, "operating_threshold.txt")},
    "val":  {"acc": float(val_metrics[0]), "prec": float(val_metrics[1]), "rec": float(val_metrics[2]),
             "f1": float(val_metrics[3]), "auc": float(val_metrics[4]), "cm": val_metrics[5]},
    "test": {"acc": float(test_metrics[0]), "prec": float(test_metrics[1]), "rec": float(test_metrics[2]),
             "f1": float(test_metrics[3]), "auc": float(test_metrics[4]), "cm": test_metrics[5]},
}
with open(os.path.join(EXP, "manifest_effnet_fold2_imagenet.json"), "w") as f: json.dump(manifest, f, indent=2)
with open(os.path.join(REP_DIR, "operating_threshold.txt"), "w") as f: f.write(f"{tau_star:.6f}\n")

print("\n✅ Saved:")
print(" - checkpoint :", manifest["paths"]["checkpoint"])
print(" - predictions:", manifest["paths"]["predictions_csv"])
print(" - manifest   :", os.path.join(EXP, 'manifest_effnet_fold2_imagenet.json'))
print(" - τ* file    :", manifest["paths"]["tau_file"])


[Warmup] Ep01 | train loss 0.7732 f1 0.154 auc 0.591 | val loss 0.6525 f1 0.800 auc 0.667
[Warmup] Ep02 | train loss 0.4348 f1 0.857 auc 0.910 | val loss 0.7168 f1 0.750 auc 0.500
[Warmup] Ep03 | train loss 0.4495 f1 0.875 auc 0.938 | val loss 0.6036 f1 0.750 auc 0.833
[Warmup] Ep04 | train loss 0.2324 f1 0.977 auc 0.992 | val loss 0.4922 f1 0.857 auc 1.000
[Warmup] Ep05 | train loss 0.2844 f1 0.833 auc 0.972 | val loss 0.4943 f1 0.857 auc 1.000
[Warmup] Ep06 | train loss 0.1788 f1 0.952 auc 0.986 | val loss 0.5801 f1 0.750 auc 1.000
[Warmup] Ep07 | train loss 0.1103 f1 0.977 auc 1.000 | val loss 0.6177 f1 0.750 auc 1.000
[Warmup] Ep08 | train loss 0.1028 f1 1.000 auc 1.000 | val loss 0.4433 f1 0.857 auc 1.000
[Warmup] Ep09 | train loss 0.1454 f1 0.950 auc 1.000 | val loss 0.2851 f1 1.000 auc 1.000
[Warmup] Ep10 | train loss 0.1383 f1 0.952 auc 0.994 | val loss 0.2369 f1 1.000 auc 1.000
[Finetune] Ep01 | train loss 0.2812 f1 0.900 auc 0.961 | val loss 0.4167 f1 0.857 auc 1.000
[Finetun

In [50]:
# EfficientNet-B0 (ImageNet) on Kaggle_CV/fold_2 — train, pick τ*, evaluate, save
import os, json, numpy as np, torch, pandas as pd
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import models, transforms as T
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# ---------- Paths ----------
FOLD = 5
BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
FOLD_DIR = f"{BASE}/fold_{FOLD}"
train_csv, val_csv, test_csv = [f"{FOLD_DIR}/{x}.csv" for x in ("train","val","test")]
assert all(os.path.exists(p) for p in [train_csv,val_csv,test_csv]), "Fold-2 CSVs missing."

EXP      = os.path.join(FOLD_DIR, "effnet_fold2_imagenet")
CKPT_DIR = os.path.join(EXP, "checkpoints")
PRED_DIR = os.path.join(EXP, "predictions")
REP_DIR  = os.path.join(EXP, "reports")
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(REP_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL2ID = {"healthy":0, "cancer":1}
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)

# ---------- Data ----------
train_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.20, scale=(0.02,0.08), ratio=(0.3,3.3), value=0.0),
])
eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImgDS(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df, self.tfm = df, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(r["filepath"])
        y = LABEL2ID[str(r["label"]).lower()]
        return self.tfm(x), torch.tensor(y, dtype=torch.long)

train_ds, val_ds, test_ds = ImgDS(train_csv, train_tf), ImgDS(val_csv, eval_tf), ImgDS(test_csv, eval_tf)
BATCH=8
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

# ---------- Model ----------
try:
    effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
except Exception:
    effnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 2)
effnet.to(device)

# class-weighted CE (small set)
counts = np.bincount(train_ds.df["label"].map(LABEL2ID).values, minlength=2).astype(np.float32)
loss_w = (counts.sum() / (2*np.maximum(counts,1))).astype(np.float32)
crit = nn.CrossEntropyLoss(weight=torch.tensor(loss_w, dtype=torch.float32, device=device))

def run_epoch(loader, train=True, model=None, opt=None):
    model = model or effnet
    model.train(train)
    losses, Ys, Ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = crit(logits, yb)
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
        with torch.no_grad():
            prob = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            Ys.append(yb.detach().cpu().numpy()); Ps.append(prob)
    Y = np.concatenate(Ys) if Ys else np.array([])
    P = np.concatenate(Ps) if Ps else np.array([])
    yhat = (P>=0.5).astype(int) if len(P) else np.array([])
    pr, rc, f1, _ = precision_recall_fscore_support(Y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(Y,P)
    except: auc = float('nan')
    return float(np.mean(losses)), f1, auc, Y, P

# ---------- Phase 1: warmup (freeze features) ----------
for p in effnet.features.parameters(): p.requires_grad=False
opt = AdamW(filter(lambda p: p.requires_grad, effnet.parameters()), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
BEST, PATIENCE, noimp = -np.inf, 6, 0
best_path = os.path.join(CKPT_DIR, "efficientnet_b0_fold2_imagenet_best.pt")

for ep in range(1, 11):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Warmup] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST + 1e-6:
        BEST, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE: break

# ---------- Phase 2: finetune (unfreeze) ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
for p in effnet.features.parameters(): p.requires_grad=True
opt = AdamW(effnet.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

BEST2, noimp = -np.inf, 0
for ep in range(1, 31):
    tr_loss, tr_f1, tr_auc, _, _ = run_epoch(train_loader, True, effnet, opt)
    va_loss, va_f1, va_auc, yv, pv = run_epoch(val_loader, False, effnet)
    print(f"[Finetune] Ep{ep:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.3f} auc {tr_auc:.3f} | "
          f"val loss {va_loss:.4f} f1 {va_f1:.3f} auc {va_auc:.3f}")
    score = np.nan_to_num(va_auc, nan=va_f1)
    if score > BEST2 + 1e-6:
        BEST2, noimp = score, 0
        torch.save(effnet.state_dict(), best_path)
    else:
        noimp += 1
    sched.step(np.nan_to_num(va_auc, nan=va_f1))
    if noimp >= PATIENCE:
        print(f"Early stop at finetune epoch {ep} (best={BEST2:.3f})"); break

# ---------- Threshold selection ----------
effnet.load_state_dict(torch.load(best_path, map_location=device))
_, _, _, yv, pv = run_epoch(val_loader, False, effnet)
_, _, _, yt, pt = run_epoch(test_loader, False, effnet)

taus = np.linspace(0,1,1001)
best_f1, tau_f1 = -1, 0.5
for t in taus:
    yhat = (pv>=t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if f1 > best_f1: best_f1, tau_f1 = f1, float(t)

tau_prec = None
for t in taus:
    yhat = (pv>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yv, yhat, average='binary', zero_division=0)
    if pr >= 0.98:
        tau_prec = float(t); break

tau_star = tau_prec if tau_prec is not None else tau_f1
policy = "prec>=0.98" if tau_prec is not None else "maxF1"

def report(y, p, t, tag):
    yhat = (p>=t).astype(int)
    acc = accuracy_score(y, yhat); pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    try: auc = roc_auc_score(y, p)
    except: auc = float('nan')
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    print(f"\n[{tag} @ τ*={t:.6f}] acc={acc:.3f} prec={pr:.3f} rec={rc:.3f} f1={f1:.3f} auc={auc:.3f}")
    print("CM (rows=true [healthy,cancer], cols=pred):\n", cm)
    return acc, pr, rc, f1, auc, cm

val_metrics  = report(yv, pv, tau_star, "VAL")
test_metrics = report(yt, pt, tau_star, "TEST")

# ---------- Save artifacts ----------
def pack(df, y_true, p, split):
    out = df[["filepath","label","source"]].copy()
    out["split"]=split; out["y_true"]=y_true; out["prob_cancer"]=p
    out["y_pred_tau*"]=(p>=tau_star).astype(int)
    return out

val_pred  = pack(val_ds.df,  yv, pv, "val")
test_pred = pack(test_ds.df, yt, pt, "test")
pred_all  = pd.concat([val_pred, test_pred], ignore_index=True)
pred_csv  = os.path.join(PRED_DIR, "effnet_fold2_imagenet_val_test_predictions.csv")
pred_all.to_csv(pred_csv, index=False)

manifest = {
    "model": "EfficientNet-B0 (ImageNet init)",
    "fold": FOLD, "policy": policy, "tau_star": float(tau_star),
    "paths": {"checkpoint": os.path.join(CKPT_DIR,"efficientnet_b0_fold2_imagenet_best.pt"),
              "predictions_csv": pred_csv,
              "tau_file": os.path.join(REP_DIR, "operating_threshold.txt")},
    "val":  {"acc": float(val_metrics[0]), "prec": float(val_metrics[1]), "rec": float(val_metrics[2]),
             "f1": float(val_metrics[3]), "auc": float(val_metrics[4]), "cm": val_metrics[5]},
    "test": {"acc": float(test_metrics[0]), "prec": float(test_metrics[1]), "rec": float(test_metrics[2]),
             "f1": float(test_metrics[3]), "auc": float(test_metrics[4]), "cm": test_metrics[5]},
}
with open(os.path.join(EXP, "manifest_effnet_fold2_imagenet.json"), "w") as f: json.dump(manifest, f, indent=2)
with open(os.path.join(REP_DIR, "operating_threshold.txt"), "w") as f: f.write(f"{tau_star:.6f}\n")

print("\n✅ Saved:")
print(" - checkpoint :", manifest["paths"]["checkpoint"])
print(" - predictions:", manifest["paths"]["predictions_csv"])
print(" - manifest   :", os.path.join(EXP, 'manifest_effnet_fold2_imagenet.json'))
print(" - τ* file    :", manifest["paths"]["tau_file"])


[Warmup] Ep01 | train loss 0.5616 f1 0.766 auc 0.791 | val loss 0.6155 f1 0.500 auc 1.000
[Warmup] Ep02 | train loss 0.3491 f1 0.865 auc 0.968 | val loss 0.4493 f1 1.000 auc 1.000
[Warmup] Ep03 | train loss 0.3200 f1 0.900 auc 0.934 | val loss 0.4330 f1 0.857 auc 1.000
[Warmup] Ep04 | train loss 0.1477 f1 1.000 auc 1.000 | val loss 0.5313 f1 0.750 auc 1.000
[Warmup] Ep05 | train loss 0.1487 f1 0.977 auc 0.992 | val loss 0.5202 f1 0.750 auc 1.000
[Warmup] Ep06 | train loss 0.2315 f1 0.927 auc 0.974 | val loss 0.4296 f1 0.857 auc 1.000
[Warmup] Ep07 | train loss 0.1643 f1 0.952 auc 0.992 | val loss 0.3335 f1 1.000 auc 1.000
[Finetune] Ep01 | train loss 0.4025 f1 0.778 auc 0.960 | val loss 0.5390 f1 0.500 auc 1.000
[Finetune] Ep02 | train loss 0.2892 f1 0.923 auc 1.000 | val loss 0.4704 f1 1.000 auc 1.000
[Finetune] Ep03 | train loss 0.2543 f1 0.950 auc 1.000 | val loss 0.4606 f1 1.000 auc 1.000
[Finetune] Ep04 | train loss 0.2341 f1 0.952 auc 0.997 | val loss 0.4285 f1 1.000 auc 1.000
[F

In [51]:
import os, glob, json, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score

BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
fold_dirs = sorted([d for d in glob.glob(f"{BASE}/fold_*") if os.path.isdir(d)])

rows = []
for fd in fold_dirs:
    csvs = glob.glob(f"{fd}/**/*predictions*.csv", recursive=True)
    if not csvs:
        continue
    df = pd.read_csv(csvs[0])
    # Expect columns: ['filepath','label','source','split','y_true','prob_cancer', ...]
    rows.append(df[df['split'].str.lower().eq('test')])

test_all = pd.concat(rows, ignore_index=True)
y = test_all['y_true'].values.astype(int)
# Use each fold’s chosen τ*: if “y_pred_*” exists, prefer that. Otherwise τ=0.5 as fallback.
if 'y_pred_tau' in test_all.columns:
    yhat = test_all['y_pred_tau'].values.astype(int)
elif 'y_pred_0p5' in test_all.columns:
    yhat = test_all['y_pred_0p5'].values.astype(int)
else:
    yhat = (test_all['prob_cancer'].values >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
acc = (tp + tn) / (tp + tn + fp + fn)

# AUC on pooled test (guard if single-class)
try:
    auc = roc_auc_score(y, test_all['prob_cancer'].values)
except Exception:
    auc = float('nan')

print(f"Global TEST: acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}")
print("CM (rows=true [healthy,cancer], cols=pred):")
print(np.array([[tn, fp],[fn, tp]]))

# Inspect healthy false positives
fp_rows = test_all[(y==0) & (yhat==1)][['filepath','source','label','prob_cancer']]
print("\nHealthy false positives (filepaths + scores):")
print(fp_rows.to_string(index=False))

outdir = f"{BASE}/aggregate"
os.makedirs(outdir, exist_ok=True)
test_all.to_csv(f"{outdir}/effnet_kagglecv_test_concat.csv", index=False)
with open(f"{outdir}/effnet_kagglecv_metrics.json",'w') as f:
    json.dump(
        dict(acc=float(acc),prec=float(prec),rec=float(rec),f1=float(f1),auc=float(auc),
             cm=[[int(tn),int(fp)],[int(fn),int(tp)]]), f, indent=2)
print(f"\nSaved:\n - {outdir}/effnet_kagglecv_test_concat.csv\n - {outdir}/effnet_kagglecv_metrics.json")


Global TEST: acc=0.741  prec=0.711  rec=0.900  f1=0.794  auc=0.881
CM (rows=true [healthy,cancer], cols=pred):
[[13 11]
 [ 3 27]]

Healthy false positives (filepaths + scores):
                                                                                                                                                                               filepath            source   label  prob_cancer
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (16).png kaggle_normal_raw healthy     0.602916
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (35).png kaggle_normal_raw healthy     0.602916
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_addit

In [52]:
import os, glob, json, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score

BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
fold_dirs = sorted([d for d in glob.glob(f"{BASE}/fold_*") if os.path.isdir(d)])

def find_pred_and_tau(fold_dir):
    # predictions CSV (first match under fold)
    preds = glob.glob(f"{fold_dir}/**/*predictions*.csv", recursive=True)
    if not preds: return None, None
    pred_csv = preds[0]
    # tau* file (first match under fold)
    taus = glob.glob(f"{fold_dir}/**/reports/operating_threshold.txt", recursive=True)
    tau = None
    if taus:
        try:
            with open(taus[0],'r') as f:
                s = f.read().strip()
            # extract float (handles lines like "τ*=0.518000")
            import re
            m = re.search(r"([0-9]*\.?[0-9]+([eE][+-]?\d+)?)", s)
            if m: tau = float(m.group(1))
        except Exception:
            tau = None
    return pred_csv, tau

all_test = []
bad = []
for fd in fold_dirs:
    pred_csv, tau = find_pred_and_tau(fd)
    if not pred_csv:
        bad.append((fd, "no predictions csv"))
        continue
    df = pd.read_csv(pred_csv)
    test = df[df["split"].str.lower().eq("test")].copy()
    if test.empty:
        bad.append((fd, "no test rows"))
        continue
    if tau is None:
        bad.append((fd, "no tau*; falling back to 0.5"))
        tau = 0.5
    test["y_pred_tau"] = (test["prob_cancer"].values >= tau).astype(int)
    test["tau_used"] = tau
    all_test.append(test)

if not all_test:
    raise RuntimeError("No fold test rows found; check paths.")

test_all = pd.concat(all_test, ignore_index=True)

y = test_all["y_true"].astype(int).values
p = test_all["prob_cancer"].values
yhat = test_all["y_pred_tau"].astype(int).values

tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
acc = (tp + tn) / (tp + tn + fp + fn)

try:
    auc = roc_auc_score(y, p)
except Exception:
    auc = float("nan")

print("===== Corrected (per-fold τ*) pooled TEST metrics =====")
print(f"acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}")
print("CM (rows=true [healthy,cancer], cols=pred):")
print(np.array([[tn, fp],[fn, tp]]))

fp_rows = test_all[(test_all["y_true"]==0) & (test_all["y_pred_tau"]==1)][["filepath","source","label","prob_cancer","tau_used"]]
print("\nHealthy false positives after τ*:")
print(fp_rows.to_string(index=False) if len(fp_rows) else "(none)")

outdir = f"{BASE}/aggregate"
os.makedirs(outdir, exist_ok=True)
test_all.to_csv(f"{outdir}/effnet_kagglecv_test_concat_tauStar.csv", index=False)
with open(f"{outdir}/effnet_kagglecv_metrics_tauStar.json",'w') as f:
    json.dump(
        dict(acc=float(acc),prec=float(prec),rec=float(rec),f1=float(f1),auc=float(auc),
             cm=[[int(tn),int(fp)],[int(fn),int(tp)]]), f, indent=2)
print(f"\nSaved:\n - {outdir}/effnet_kagglecv_test_concat_tauStar.csv\n - {outdir}/effnet_kagglecv_metrics_tauStar.json")
if bad:
    print("\nNotes (folds needing attention):")
    for b in bad: print(" -", b)


===== Corrected (per-fold τ*) pooled TEST metrics =====
acc=0.815  prec=0.750  rec=1.000  f1=0.857  auc=0.881
CM (rows=true [healthy,cancer], cols=pred):
[[14 10]
 [ 0 30]]

Healthy false positives after τ*:
                                                                                                                                                                               filepath            source   label  prob_cancer  tau_used
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (16).png kaggle_normal_raw healthy     0.602916     0.000
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (35).png kaggle_normal_raw healthy     0.602916     0.000
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PrePr

In [53]:
import os, glob, json, re, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

BASE = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
fold_dirs = sorted([d for d in glob.glob(f"{BASE}/fold_*") if os.path.isdir(d)])

def read_tau_file(path):
    try:
        with open(path, "r") as f:
            s = f.read()
        m = re.search(r"([0-9]*\.?[0-9]+([eE][+-]?\d+)?)", s)
        if not m:
            return None
        tau = float(m.group(1))
        # sanity: tau must be in (0,1)
        if not (0.0 < tau < 1.0):
            return None
        return tau
    except Exception:
        return None

def pick_tau_from_val(val_df, prec_target=0.98):
    """Choose tau that achieves precision≥prec_target on VAL if possible, else max-F1."""
    y = val_df["y_true"].astype(int).values
    p = val_df["prob_cancer"].astype(float).values
    # candidate thresholds (include unique probs + edges)
    taus = np.unique(np.concatenate([p, [0.0, 1.0]]))
    taus.sort()
    best_for_prec = None
    for t in taus:
        yhat = (p >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
        if pr >= prec_target:
            if best_for_prec is None or rc > best_for_prec[1] or (rc == best_for_prec[1] and f1 > best_for_prec[2]):
                best_for_prec = (t, rc, f1)
    if best_for_prec is not None:
        return float(best_for_prec[0])

    # fallback: max-F1
    best_t, best_f1 = 0.5, -1.0
    for t in taus:
        yhat = (p >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

all_test = []
notes = []
for fd in fold_dirs:
    # predictions csv
    preds = glob.glob(f"{fd}/**/*predictions*.csv", recursive=True)
    if not preds:
        notes.append((fd, "no predictions csv"))
        continue
    df = pd.read_csv(preds[0])
    df["split"] = df["split"].astype(str).str.lower()

    val = df[df["split"]=="val"].copy()
    test = df[df["split"]=="test"].copy()
    if val.empty or test.empty:
        notes.append((fd, f"val empty? {val.empty} | test empty? {test.empty}"))
        continue

    # parse tau*
    tau_files = glob.glob(f"{fd}/**/reports/operating_threshold.txt", recursive=True)
    tau = read_tau_file(tau_files[0]) if tau_files else None
    if tau is None:
        # recompute from val
        tau = pick_tau_from_val(val, prec_target=0.98)
        src = "VAL-derived"
    else:
        src = "file"
    test["y_pred_tau"] = (test["prob_cancer"].astype(float).values >= tau).astype(int)
    test["tau_used"] = tau
    test["tau_source"] = src
    all_test.append(test)

if not all_test:
    raise RuntimeError("No test rows collected from folds.")

test_all = pd.concat(all_test, ignore_index=True)

y = test_all["y_true"].astype(int).values
p = test_all["prob_cancer"].astype(float).values
yhat = test_all["y_pred_tau"].astype(int).values

tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
acc = (tp + tn) / (tp + tn + fp + fn)
try:
    auc = roc_auc_score(y, p)
except Exception:
    auc = float("nan")

print("===== Corrected (robust τ*) pooled TEST metrics =====")
print(f"acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}")
print("CM (rows=true [healthy,cancer], cols=pred):")
print(np.array([[tn, fp],[fn, tp]]))

fps = test_all[(test_all["y_true"]==0) & (test_all["y_pred_tau"]==1)][
    ["filepath","source","label","prob_cancer","tau_used","tau_source"]
]
print("\nHealthy false positives after robust τ*:")
print(fps.to_string(index=False) if len(fps) else "(none)")

outdir = f"{BASE}/aggregate"
os.makedirs(outdir, exist_ok=True)
test_all.to_csv(f"{outdir}/effnet_kagglecv_test_concat_tauStar_robust.csv", index=False)
with open(f"{outdir}/effnet_kagglecv_metrics_tauStar_robust.json","w") as f:
    json.dump(
        dict(acc=float(acc),prec=float(prec),rec=float(rec),f1=float(f1),auc=float(auc),
             cm=[[int(tn),int(fp)],[int(fn),int(tp)]]), f, indent=2)
print(f"\nSaved:\n - {outdir}/effnet_kagglecv_test_concat_tauStar_robust.csv\n - {outdir}/effnet_kagglecv_metrics_tauStar_robust.json")
if notes:
    print("\nNotes:")
    for n in notes: print(" -", n)


===== Corrected (robust τ*) pooled TEST metrics =====
acc=0.796  prec=0.771  rec=0.900  f1=0.831  auc=0.881
CM (rows=true [healthy,cancer], cols=pred):
[[16  8]
 [ 3 27]]

Healthy false positives after robust τ*:
                                                                                                                                                                               filepath            source   label  prob_cancer  tau_used  tau_source
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (36).png kaggle_normal_raw healthy     0.602916  0.602916 VAL-derived
 /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (5).png kaggle_normal_raw healthy     0.602917  0.602916 VAL-derived
/workspaces/cmp9137-advanced-machine-learning/CMP913

In [54]:
import glob, os, json, math
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score, confusion_matrix
)

# --- Config: where your per-fold CSVs are ---
ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
PATTERN = os.path.join(ROOT, "fold_*", "*imagenet", "predictions", "*_val_test_predictions.csv")

out_dir = os.path.join(ROOT, "aggregate_calibrated")
os.makedirs(out_dir, exist_ok=True)

def sigmoid(x): return 1.0/(1.0+np.exp(-x))
def logit(p, eps=1e-6): 
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def fit_temperature(val_probs, y_true, epochs=200, lr=0.05):
    # val_probs are probabilities for the positive (cancer) class
    # Convert to logits, then learn a scalar T minimizing NLL
    logits = torch.tensor(logit(val_probs), dtype=torch.float32)
    targets= torch.tensor(y_true.astype(np.float32))
    T = torch.nn.Parameter(torch.tensor(1.0))
    opt = torch.optim.LBFGS([T], lr=lr, max_iter=epochs, line_search_fn="strong_wolfe")

    def nll_of(temp):
        # Calibrated logits = logits / T
        z = logits / temp
        # Stable BCE with logits
        return torch.nn.functional.binary_cross_entropy_with_logits(z, targets)

    def closure():
        opt.zero_grad()
        loss = nll_of(T)
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        t = float(torch.clamp(T, 1e-3, 100.0))
    return t

def pick_threshold(probs, y, target_prec=0.98):
    # Try precision constraint first; if impossible, fallback to max-F1
    ths = np.unique(np.round(probs, 6))
    ths = np.concatenate([ths, np.linspace(0,1,401)])
    ths = np.unique(ths)
    best = {"t":0.5, "f1":-1, "prec":0, "rec":0}
    candidate_t = None
    for t in ths:
        yhat = (probs >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
        if pr >= target_prec:
            candidate_t = (t, pr, rc, f1)
            break
        if f1 > best["f1"]:
            best = {"t":t, "f1":f1, "prec":pr, "rec":rc}
    if candidate_t is not None:
        return candidate_t[0], {"policy": f"prec>={target_prec}", "prec":candidate_t[1], "rec":candidate_t[2], "f1":candidate_t[3]}
    else:
        return best["t"], {"policy": "maxF1", "prec":best["prec"], "rec":best["rec"], "f1":best["f1"]}

def metrics_at(y, p, t):
    yhat = (p>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    acc = (yhat==y).mean()
    try: auc = roc_auc_score(y, p)
    except: auc = float("nan")
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    return dict(acc=float(acc), prec=float(pr), rec=float(rc), f1=float(f1), auc=float(auc), cm=cm, tau=float(t))

# --- Load all folds ---
csvs = sorted(glob.glob(PATTERN))
assert csvs, f"No per-fold CSVs found with pattern: {PATTERN}"

rows = []
per_fold = {}

for path in csvs:
    df = pd.read_csv(path)
    # Expect columns: filepath,label,source,split,prob_cancer,y_true (your previous format)
    assert {"split","prob_cancer","y_true"}.issubset(df.columns)
    val = df[df.split=="val"].copy()
    tst = df[df.split=="test"].copy()

    yv, pv = val.y_true.values.astype(int), val.prob_cancer.values.astype(float)
    yt, pt = tst.y_true.values.astype(int), tst.prob_cancer.values.astype(float)

    # Fit temperature on VAL
    T = fit_temperature(pv, yv)
    pv_cal = sigmoid(logit(pv)/T)
    pt_cal = sigmoid(logit(pt)/T)

    # Pick threshold on VAL (after calibration)
    tau, pol = pick_threshold(pv_cal, yv, target_prec=0.98)

    # Eval on TEST
    m_val = metrics_at(yv, pv_cal, tau)
    m_tst = metrics_at(yt, pt_cal, tau)

    per_fold[os.path.dirname(os.path.dirname(path))] = {
        "csv": path, "T": T, "tau": tau, "policy": pol, 
        "val": m_val, "test": m_tst,
        "counts": {"val": len(val), "test": len(tst)}
    }

    # Keep calibrated test rows for pooled summary
    dft = tst.copy()
    dft["prob_cancer_cal"] = pt_cal
    dft["tau_used"] = tau
    dft["T"] = T
    rows.append(dft)

# --- Pooled TEST (calibrated) with per-fold τ* ---
pooled = pd.concat(rows, ignore_index=True)
pooled["yhat"] = (pooled["prob_cancer_cal"] >= pooled["tau_used"]).astype(int)
yt = pooled["y_true"].values.astype(int)
pt = pooled["prob_cancer_cal"].values.astype(float)
t_used = pooled["tau_used"].values

pr, rc, f1, _ = precision_recall_fscore_support(yt, pooled["yhat"].values, average='binary', zero_division=0)
acc = (pooled["yhat"].values==yt).mean()
try: auc = roc_auc_score(yt, pt)
except: auc = float("nan")
cm  = confusion_matrix(yt, pooled["yhat"].values, labels=[0,1]).tolist()

summary = {"acc":float(acc), "prec":float(pr), "rec":float(rc), "f1":float(f1), "auc":float(auc), "cm":cm}
print("=== Calibrated pooled TEST (per-fold τ*) ===")
print(summary)

# Save artifacts
pooled_out = os.path.join(out_dir, "effnet_kagglecv_test_concat_tauStar_CALIBRATED.csv")
pooled.to_csv(pooled_out, index=False)
with open(os.path.join(out_dir, "effnet_kagglecv_metrics_tauStar_CALIBRATED.json"), "w") as f:
    json.dump({"per_fold":per_fold, "pooled_test":summary}, f, indent=2)

# Also print the remaining healthy false positives (if any)
fp = pooled[(pooled["label"].astype(str)=="healthy") & (pooled["yhat"]==1)]
if len(fp):
    print("\nHealthy FPs after calibration (top 10 by prob):")
    disp = fp.sort_values("prob_cancer_cal", ascending=False)[["filepath","source","prob_cancer_cal","tau_used","T"]].head(10)
    print(disp.to_string(index=False))
else:
    print("\nNo healthy false positives after calibration.")
print(f"\nSaved CSV -> {pooled_out}")


=== Calibrated pooled TEST (per-fold τ*) ===
{'acc': 0.9074074074074074, 'prec': 0.8571428571428571, 'rec': 1.0, 'f1': 0.9230769230769231, 'auc': 0.9583333333333333, 'cm': [[19, 5], [0, 30]]}

Healthy FPs after calibration (top 10 by prob):
                                                                                                                                                                               filepath            source  prob_cancer_cal  tau_used        T
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (53).png kaggle_normal_raw         0.722664  0.656571 0.387421
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (40).png kaggle_normal_raw         0.687999  0.629282 1.344708
/workspaces/cmp9137-advanced-machine-learning

In [55]:
import glob, os, json, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
PATTERN = os.path.join(ROOT, "fold_*", "*imagenet", "predictions", "*_val_test_predictions.csv")
OUTDIR = os.path.join(ROOT, "aggregate_platt")
os.makedirs(OUTDIR, exist_ok=True)

def sigmoid(x): return 1/(1+np.exp(-x))
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def pick_threshold(probs, y, target_prec=0.98):
    ths = np.unique(np.r_[np.round(probs,6), np.linspace(0,1,401)])
    best = {"t":0.5, "f1":-1, "prec":0, "rec":0}
    chosen = None
    for t in ths:
        yhat = (probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
        if pr >= target_prec:
            chosen = (t, pr, rc, f1); break
        if f1 > best["f1"]:
            best = {"t":t, "f1":f1, "prec":pr, "rec":rc}
    if chosen: return chosen[0], {"policy": f"prec>={target_prec}", "prec":chosen[1], "rec":chosen[2], "f1":chosen[3]}
    return best["t"], {"policy":"maxF1", "prec":best["prec"], "rec":best["rec"], "f1":best["f1"]}

def metrics_at(y, p, t):
    yhat = (p>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    acc = float((yhat==y).mean())
    try: auc = float(roc_auc_score(y, p))
    except: auc = float("nan")
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    return dict(acc=acc, prec=float(pr), rec=float(rc), f1=float(f1), auc=auc, cm=cm, tau=float(t))

csvs = sorted(glob.glob(PATTERN))
assert csvs, f"No per-fold CSVs found at: {PATTERN}"

pooled_rows, per_fold = [], {}
for path in csvs:
    df = pd.read_csv(path)
    assert {"split","prob_cancer","y_true"}.issubset(df.columns)
    val = df[df.split=="val"].copy()
    tst = df[df.split=="test"].copy()

    yv = val.y_true.values.astype(int)
    yt = tst.y_true.values.astype(int)
    pv = val.prob_cancer.values.astype(float)
    pt = tst.prob_cancer.values.astype(float)

    Xv = logit(pv).reshape(-1,1)
    Xt = logit(pt).reshape(-1,1)

    # Platt scaling (1D logistic)
    clf = LogisticRegression(C=1.0, solver="lbfgs", max_iter=200)
    clf.fit(Xv, yv)
    pv_cal = clf.predict_proba(Xv)[:,1]
    pt_cal = clf.predict_proba(Xt)[:,1]

    tau, pol = pick_threshold(pv_cal, yv, target_prec=0.98)
    m_val = metrics_at(yv, pv_cal, tau)
    m_tst = metrics_at(yt, pt_cal, tau)

    key = os.path.dirname(os.path.dirname(path))
    per_fold[key] = {
        "csv": path,
        "coef": float(clf.coef_[0,0]),
        "intercept": float(clf.intercept_[0]),
        "tau": float(tau),
        "policy": pol,
        "val": m_val,
        "test": m_tst,
        "counts": {"val":len(val), "test":len(tst)}
    }

    dft = tst.copy()
    dft["prob_cancer_cal_platt"] = pt_cal
    dft["tau_used"] = tau
    dft["coef"] = float(clf.coef_[0,0])
    dft["intercept"] = float(clf.intercept_[0])
    pooled_rows.append(dft)

pooled = pd.concat(pooled_rows, ignore_index=True)
pooled["yhat"] = (pooled["prob_cancer_cal_platt"] >= pooled["tau_used"]).astype(int)
yt = pooled["y_true"].values.astype(int)
pt = pooled["prob_cancer_cal_platt"].values.astype(float)

pr, rc, f1, _ = precision_recall_fscore_support(yt, pooled["yhat"].values, average='binary', zero_division=0)
acc = float((pooled["yhat"].values==yt).mean())
try: auc = float(roc_auc_score(yt, pt))
except: auc = float("nan")
cm  = confusion_matrix(yt, pooled["yhat"].values, labels=[0,1]).tolist()
summary = {"acc":acc, "prec":float(pr), "rec":float(rc), "f1":float(f1), "auc":auc, "cm":cm}

print("=== Platt-calibrated pooled TEST (per-fold τ*) ===")
print(summary)

pooled_csv = os.path.join(OUTDIR, "effnet_kagglecv_test_concat_tauStar_PLATT.csv")
pooled.to_csv(pooled_csv, index=False)
with open(os.path.join(OUTDIR, "effnet_kagglecv_metrics_tauStar_PLATT.json"), "w") as f:
    json.dump({"per_fold":per_fold, "pooled_test":summary}, f, indent=2)

fp = pooled[(pooled["label"].astype(str)=="healthy") & (pooled["yhat"]==1)]
if len(fp):
    print("\nHealthy FPs after Platt (top 10):")
    print(fp.sort_values("prob_cancer_cal_platt", ascending=False)[["filepath","source","prob_cancer_cal_platt","tau_used"]].head(10).to_string(index=False))
else:
    print("\nNo healthy FPs after Platt.")
print(f"\nSaved CSV -> {pooled_csv}")


=== Platt-calibrated pooled TEST (per-fold τ*) ===
{'acc': 0.9074074074074074, 'prec': 0.8571428571428571, 'rec': 1.0, 'f1': 0.9230769230769231, 'auc': 0.9263888888888889, 'cm': [[19, 5], [0, 30]]}

Healthy FPs after Platt (top 10):
                                                                                                                                                                               filepath            source  prob_cancer_cal_platt  tau_used
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (38).png kaggle_normal_raw               0.714151  0.554105
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (28).png kaggle_normal_raw               0.714103  0.554105
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced

In [56]:
import os, glob, json, numpy as np, pandas as pd, torch
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms as T, models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
PATTERN = os.path.join(ROOT, "fold_*", "*imagenet", "predictions", "*_val_test_predictions.csv")
OUTDIR = os.path.join(ROOT, "aggregate_tta_platt")
os.makedirs(OUTDIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- model loader (EffNet-B0, 2 classes) ---
def load_effnet(ckpt_path):
    m = models.efficientnet_b0(weights=None)
    m.classifier[1] = torch.nn.Linear(m.classifier[1].in_features, 2)
    sd = torch.load(ckpt_path, map_location="cpu")
    m.load_state_dict(sd)
    m.eval().to(device)
    return m

# --- I/O & transforms ---
to_tensor = T.ToTensor()
normalize = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def to3(img):  # ensure 3-ch from L
    if img.mode != "L":
        img = img.convert("L")
    x = to_tensor(img)          # (1,H,W)
    x = x.repeat(3,1,1)         # (3,H,W)
    return normalize(x)

def tta_variants(img, pad=8, size=224):
    # 5 crops: center + shifts (±pad horizontally and vertically)
    # Implement via pad then crop windows at fixed offsets
    base = T.Pad(pad, padding_mode='edge')(img)
    W, H = base.size
    cx, cy = W//2, H//2
    offsets = [(0,0), (pad,0), (-pad,0), (0,pad), (0,-pad)]
    crops = []
    for dx,dy in offsets:
        left   = cx - size//2 + dx
        top    = cy - size//2 + dy
        right  = left + size
        bottom = top  + size
        crop = base.crop((left, top, right, bottom))
        crops.append(crop)
    return crops

@torch.no_grad()
def predict_tta(model, paths, batch=32):
    probs = []
    for i in range(0, len(paths), batch):
        chunk = paths[i:i+batch]
        xs = []
        for p in chunk:
            img = Image.open(p)
            views = [to3(c) for c in tta_variants(img)]
            x = torch.stack(views, 0)              # (V,3,224,224)
            xs.append(x)
        xb = torch.stack([x.mean(0) for x in xs], 0).to(device)  # mean over 5 views (simple & fast)
        logits = model(xb)
        p = F.softmax(logits, dim=1)[:,1].cpu().numpy()
        probs.append(p)
    return np.concatenate(probs) if probs else np.array([])

def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def pick_threshold(probs, y, target_prec=0.98):
    ths = np.unique(np.r_[np.round(probs,6), np.linspace(0,1,401)])
    best_t, best_f1, best_tuple = 0.5, -1, None
    for t in ths:
        yhat = (probs>=t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
        if pr >= target_prec:
            return float(t), {"policy": f"prec>={target_prec}", "prec":float(pr), "rec":float(rc), "f1":float(f1)}
        if f1 > best_f1:
            best_f1, best_t = f1, t
    yhat = (probs>=best_t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    return float(best_t), {"policy":"maxF1", "prec":float(pr), "rec":float(rc), "f1":float(f1)}

def metrics_at(y, p, t):
    yhat = (p>=t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    acc = float((yhat==y).mean())
    try: auc = float(roc_auc_score(y, p))
    except: auc = float("nan")
    cm = confusion_matrix(y, yhat, labels=[0,1]).tolist()
    return dict(acc=acc, prec=float(pr), rec=float(rc), f1=float(f1), auc=auc, cm=cm, tau=float(t))

# --- discover folds ---
csvs = sorted(glob.glob(PATTERN))
assert csvs, f"No per-fold CSVs found at: {PATTERN}"

pooled_rows, per_fold = [], {}
for csv in csvs:
    base = os.path.dirname(os.path.dirname(csv))   # .../fold_X/effnet_foldY_imagenet
    ckpts = glob.glob(os.path.join(base, "checkpoints", "*.pt"))
    assert ckpts, f"No checkpoint under {base}/checkpoints"
    ckpt = ckpts[0]
    model = load_effnet(ckpt)

    df = pd.read_csv(csv)
    val = df[df.split=="val"].copy()
    tst = df[df.split=="test"].copy()

    yv = val["y_true"].astype(int).values
    yt = tst["y_true"].astype(int).values
    pv_tta = predict_tta(model, val["filepath"].tolist())
    pt_tta = predict_tta(model, tst["filepath"].tolist())

    # Platt on VAL logits
    clf = LogisticRegression(C=1.0, solver="lbfgs", max_iter=200)
    Xv = logit(pv_tta).reshape(-1,1); Xt = logit(pt_tta).reshape(-1,1)
    clf.fit(Xv, yv)
    pv_cal = clf.predict_proba(Xv)[:,1]
    pt_cal = clf.predict_proba(Xt)[:,1]

    tau, pol = pick_threshold(pv_cal, yv, target_prec=0.98)
    m_val = metrics_at(yv, pv_cal, tau)
    m_tst = metrics_at(yt, pt_cal, tau)

    key = base
    per_fold[key] = {
        "csv": csv, "ckpt": ckpt,
        "coef": float(clf.coef_[0,0]), "intercept": float(clf.intercept_[0]),
        "tau": float(tau), "policy": pol, "val": m_val, "test": m_tst,
        "counts": {"val":len(val), "test":len(tst)}
    }

    dft = tst.copy()
    dft["prob_cancer_tta_platt"] = pt_cal
    dft["tau_used"] = tau
    pooled_rows.append(dft)

# --- pooled summary ---
pooled = pd.concat(pooled_rows, ignore_index=True)
pooled["yhat"] = (pooled["prob_cancer_tta_platt"] >= pooled["tau_used"]).astype(int)
yt = pooled["y_true"].astype(int).values
pt = pooled["prob_cancer_tta_platt"].astype(float).values

pr, rc, f1, _ = precision_recall_fscore_support(yt, pooled["yhat"], average='binary', zero_division=0)
acc = float((pooled["yhat"]==yt).mean())
try: auc = float(roc_auc_score(yt, pt))
except: auc = float("nan")
cm  = confusion_matrix(yt, pooled["yhat"], labels=[0,1]).tolist()
summary = {"acc":acc, "prec":float(pr), "rec":float(rc), "f1":float(f1), "auc":auc, "cm":cm}

print("=== TTA+Platt pooled TEST (per-fold τ*) ===")
print(summary)

# show remaining healthy FPs
fp = pooled[(pooled["label"].astype(str)=="healthy") & (pooled["yhat"]==1)]
if len(fp):
    print("\nHealthy FPs after TTA+Platt (top 10):")
    cols = ["filepath","source","prob_cancer_tta_platt","tau_used"]
    print(fp.sort_values("prob_cancer_tta_platt", ascending=False)[cols].head(10).to_string(index=False))
else:
    print("\nNo healthy FPs after TTA+Platt.")

# save
pooled_csv = os.path.join(OUTDIR, "effnet_kagglecv_test_concat_tauStar_TTA_PLATT.csv")
pooled.to_csv(pooled_csv, index=False)
with open(os.path.join(OUTDIR, "effnet_kagglecv_metrics_tauStar_TTA_PLATT.json"), "w") as f:
    json.dump({"per_fold":per_fold, "pooled_test":summary}, f, indent=2)
print(f"\nSaved CSV -> {pooled_csv}")


=== TTA+Platt pooled TEST (per-fold τ*) ===
{'acc': 0.7222222222222222, 'prec': 0.7272727272727273, 'rec': 0.8, 'f1': 0.7619047619047619, 'auc': 0.8569444444444444, 'cm': [[15, 9], [6, 24]]}

Healthy FPs after TTA+Platt (top 10):
                                                                                                                                                                               filepath            source  prob_cancer_tta_platt  tau_used
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (38).png kaggle_normal_raw               0.716130  0.505000
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle/Normal 1 (28).png kaggle_normal_raw               0.716130  0.505000
/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Ma

In [57]:
# --- Corrected TTA + Platt calibration (per-fold) and pooled metrics ---
# Works with your fold folders like:
# Kaggle_CV/fold_1/effnet_fold1_imagenet/
# Kaggle_CV/fold_2/effnet_fold2_imagenet/ (and so on)

import os, glob, json, numpy as np, pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torchvision import models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# -------- paths --------
ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
OUT  = os.path.join(ROOT, "aggregate_tta_platt_fixed")
os.makedirs(OUT, exist_ok=True)

# -------- image -> tensor helpers --------
IM_SIZE = 224
IMNET_MEAN = [0.485, 0.456, 0.406]
IMNET_STD  = [0.229, 0.224, 0.225]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def to3(img: Image.Image):
    """PIL -> 3ch normalized tensor (ImageNet stats)."""
    # Resize to be safe
    if img.size != (IM_SIZE, IM_SIZE):
        img = img.resize((IM_SIZE, IM_SIZE), Image.BICUBIC)
    t = TF.to_tensor(img)  # (C,H,W)
    if t.shape[0] == 1:
        t = t.repeat(3, 1, 1)
    t = TF.normalize(t, IMNET_MEAN, IMNET_STD)
    return t

def tta_variants(img: Image.Image):
    """Five fixed crops (center+corners). If img is 224, these may repeat (that's ok)."""
    # Work on a slightly larger canvas to make crops meaningful
    base = img.resize((256, 256), Image.BICUBIC)
    crops = TF.five_crop(base, IM_SIZE)  # tuple of 5 PIL images
    return [to3(c) for c in crops]       # list of 5 tensors (3,224,224)

@torch.no_grad()
def predict_tta(model: nn.Module, paths, batch=16, agg="mean"):
    """
    Run TTA correctly: run model on each view, then aggregate PROBABILITIES (mean/median) per image.
    Returns np.array of p(cancer).
    """
    model.eval()
    out = []
    for i in range(0, len(paths), batch):
        chunk = paths[i:i+batch]
        # Build a big batch: (len(chunk)*V, 3, 224, 224)
        view_batches = []
        for p in chunk:
            img = Image.open(p).convert("RGB")  # robust read
            views = tta_variants(img)           # list of 5 tensors
            view_batches.append(torch.stack(views, 0))  # (V,3,224,224)
        V = view_batches[0].shape[0]
        xb = torch.cat(view_batches, 0).to(device)
        logits = model(xb)
        probs  = torch.softmax(logits, dim=1)[:, 1]     # p(cancer)
        probs  = probs.view(len(chunk), V).cpu().numpy()
        p_img  = probs.mean(1) if agg == "mean" else np.median(probs, axis=1)
        out.append(p_img)
    return np.concatenate(out) if out else np.array([])

# -------- model loader (EfficientNet-B0 w/ 2 classes) --------
def load_effnet_b0(checkpoint_path: str) -> nn.Module:
    m = models.efficientnet_b0(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    sd = torch.load(checkpoint_path, map_location="cpu")
    # allow for checkpoints saved via state_dict or wrapped dicts
    if isinstance(sd, dict) and "model_state" in sd:
        sd = sd["model_state"]
    m.load_state_dict(sd, strict=False)
    m.to(device).eval()
    return m

# -------- metrics helper --------
def metrics_at_threshold(y_true, p, t):
    yhat = (p >= t).astype(int)
    acc  = accuracy_score(y_true, yhat)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, yhat, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y_true, p)
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, yhat, labels=[0,1]).tolist()
    return {"acc":acc, "prec":pr, "rec":rc, "f1":f1, "auc":auc, "cm":cm}

# -------- fold discovery --------
# We will look for any '*imagenet' subfolder under fold_* (works with your naming variations).
FOLD_RUNS = sorted(glob.glob(os.path.join(ROOT, "fold_*", "*imagenet")))
assert FOLD_RUNS, f"No fold runs found under {ROOT}"

# -------- main loop per fold --------
concat_rows = []
pooled_y, pooled_p, pooled_taus = [], [], []

print("Found fold runs:")
for fr in FOLD_RUNS:
    print(" -", fr)

for fr in FOLD_RUNS:
    # locate files
    pred_csv = glob.glob(os.path.join(fr, "predictions", "*val_test_predictions.csv"))
    ckpt     = glob.glob(os.path.join(fr, "checkpoints", "*.pt"))
    tau_txt  = os.path.join(fr, "reports", "operating_threshold.txt")
    assert pred_csv and ckpt and os.path.exists(tau_txt), f"Missing files in {fr}"
    pred_csv, ckpt = pred_csv[0], ckpt[0]

    # load τ*
    with open(tau_txt, "r") as f:
        s = f.read().strip()
    # parse float from the line, e.g. "τ*=0.671000"
    import re
    m = re.search(r"([0-9]*\.?[0-9]+([eE][-+]?\d+)?)", s)
    tau_star = float(m.group(1)) if m else 0.5

    # load fold lists from the predictions CSV (has filepaths & y_true)
    df = pd.read_csv(pred_csv)
    val = df[df["split"]=="val"].copy()
    tst = df[df["split"]=="test"].copy()
    yv  = val["y_true"].values.astype(int)
    yt  = tst["y_true"].values.astype(int)

    # model
    model = load_effnet_b0(ckpt)

    # -------- TTA on VAL/TEST (aggregate predictions, not pixels) --------
    pv_tta = predict_tta(model, val["filepath"].tolist(), agg="mean")
    pt_tta = predict_tta(model, tst["filepath"].tolist(), agg="mean")

    # -------- Platt calibration on VAL; apply to TEST --------
    lr = LogisticRegression(solver="lbfgs")
    lr.fit(pv_tta.reshape(-1,1), yv)
    pv_cal = lr.predict_proba(pv_tta.reshape(-1,1))[:,1]
    pt_cal = lr.predict_proba(pt_tta.reshape(-1,1))[:,1]

    # per-fold metrics @ τ*
    val_metrics = metrics_at_threshold(yv, pv_cal, tau_star)
    test_metrics = metrics_at_threshold(yt, pt_cal, tau_star)
    print(f"\n[{os.path.basename(os.path.dirname(fr))}] τ*={tau_star:.6f}")
    print("VAL :", val_metrics)
    print("TEST:", test_metrics)

    # collect pooled
    pooled_y.append(yt)
    pooled_p.append(pt_cal)
    pooled_taus.append(np.full_like(yt, fill_value=tau_star, dtype=float))

    # stash for CSV
    dft = tst[["filepath","source","label","y_true"]].copy()
    dft["prob_cancer_tta_platt"] = pt_cal
    dft["tau_used"] = tau_star
    dft["fold_dir"] = fr
    concat_rows.append(dft)

# -------- pooled metrics (per-fold τ*) --------
pooled_y = np.concatenate(pooled_y) if pooled_y else np.array([])
pooled_p = np.concatenate(pooled_p) if pooled_p else np.array([])
pooled_t = np.concatenate(pooled_taus) if pooled_taus else np.array([])

yhat_pool = (pooled_p >= pooled_t).astype(int)
acc  = accuracy_score(pooled_y, yhat_pool)
pr, rc, f1, _ = precision_recall_fscore_support(pooled_y, yhat_pool, average="binary", zero_division=0)
try:
    auc = roc_auc_score(pooled_y, pooled_p)
except Exception:
    auc = float("nan")
cm = confusion_matrix(pooled_y, yhat_pool, labels=[0,1]).tolist()
pooled_metrics = {"acc":acc, "prec":pr, "rec":rc, "f1":f1, "auc":auc, "cm":cm}

# -------- save outputs --------
agg_csv = os.path.join(OUT, "effnet_kagglecv_test_concat_tauStar_TTA_PLATT_FIXED.csv")
pd.concat(concat_rows, ignore_index=True).to_csv(agg_csv, index=False)

agg_json = os.path.join(OUT, "effnet_kagglecv_metrics_tauStar_TTA_PLATT_FIXED.json")
with open(agg_json, "w") as f:
    json.dump(pooled_metrics, f, indent=2)

print("\n=== TTA+Platt (fixed) pooled TEST (per-fold τ*) ===")
print(pooled_metrics)
print("\nSaved:\n -", agg_csv, "\n -", agg_json)


Found fold runs:
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_1/effnet_fold1_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_2/effnet_fold2_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_3/effnet_fold2_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_4/effnet_fold2_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_5/effnet_fold2_imagenet

[fold_1] τ*=0.518000
VAL : {'acc': 0.6, 'prec': 0.6, 'rec': 1.0, 'f1': 0.75, 'auc': 1.0, 'cm': [[0, 2], [0, 3]]}
TEST: {'acc': 0.5454545454545454, 'prec': 0.5454545454545454, 'rec': 1.0, 'f1': 0.7058823529411765, 'auc': 0.9666666666666668, 'cm': [[0, 5], [0, 6]]}

[fold_2] τ*=0.671000
VAL : {'acc': 0.4, 'prec': 0.0, 'rec': 0.0, 'f1': 0.0, 'auc': 1.0, 'cm': [[2, 0], [3, 

In [58]:
# recalibrate_and_pool_kagglecv.py
import os, glob, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression

ROOT = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV"
OUT  = os.path.join(ROOT, "aggregate_tauCAL")  # new output dir to avoid overwrite
os.makedirs(OUT, exist_ok=True)

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
EPS = 1e-6

def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1.0 - p))

def platt_calibrate_fit(y_val, p_val):
    """
    Fit Platt scaling: p_cal = sigmoid(a*logit(p) + b)
    Returns (a, b). Falls back to identity if fitting fails.
    """
    try:
        z = logit(p_val).reshape(-1, 1)
        # Small regularization to avoid perfect separation issues on tiny sets
        lr = LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs", max_iter=1000
        )
        lr.fit(z, y_val.astype(int))
        # For 2-class LR, decision function = coef * z + intercept
        a = float(lr.coef_[0, 0])
        b = float(lr.intercept_[0])
        return a, b
    except Exception as e:
        warnings.warn(f"Platt fit failed ({e}); using identity calibration.")
        return 1.0, 0.0  # identity: sigmoid(logit(p)) == p

def platt_apply(p, a, b):
    z = logit(p)
    s = 1.0 / (1.0 + np.exp(-(a * z + b)))
    return np.clip(s, 0.0, 1.0)

def metrics_at_threshold(y, p, t):
    yhat = (p >= t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y, yhat, average="binary", zero_division=0
    )
    try:
        auc = roc_auc_score(y, p)
    except Exception:
        auc = float("nan")
    acc = (yhat == y).mean()
    cm = confusion_matrix(y, yhat, labels=[0, 1]).tolist()
    return {
        "acc": float(acc), "prec": float(pr), "rec": float(rc),
        "f1": float(f1), "auc": float(auc), "cm": cm
    }

def pick_tau_on_val(y_true, p_val, prec_target=0.98):
    """
    Choose τ on VAL after calibration:
      1) Best recall among thresholds with precision ≥ prec_target
      2) Fallback to max-F1 if precision target unattainable
    """
    thr = np.unique(np.r_[0.0, p_val, 1.0])
    cand = []
    for t in thr:
        yhat = (p_val >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(
            y_true, yhat, average="binary", zero_division=0
        )
        cand.append((t, pr, rc, f1))
    ok = [(t, pr, rc, f1) for (t, pr, rc, f1) in cand if pr >= prec_target]
    if ok:
        ok.sort(key=lambda x: (x[2], -x[0]), reverse=True)  # max recall, then smaller τ
        tau = ok[0][0]
        policy = f"prec>={prec_target:.2f}"
    else:
        tau = max(cand, key=lambda x: x[3])[0]  # max F1
        policy = "maxF1"
    return float(tau), policy

def load_fold_prediction_csvs(root=ROOT):
    """
    Find all per-fold predictions CSVs that contain VAL+TEST rows.
    Pattern-agnostic: looks for **/predictions/*val_test_predictions.csv
    """
    pattern = os.path.join(root, "fold_*", "**", "predictions", "*val_test_predictions.csv")
    files = sorted(glob.glob(pattern, recursive=True))
    return files

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
files = load_fold_prediction_csvs(ROOT)
if not files:
    raise FileNotFoundError("No per-fold predictions found under Kaggle_CV/*/predictions/*val_test_predictions.csv")

print("Found fold runs:")
for f in files:
    print(" -", Path(f).parent.parent.as_posix())

pooled_y, pooled_p, pooled_taus = [], [], []
concat_rows = []
per_fold_summary = []

for csv_path in files:
    fold_dir = Path(csv_path).parent.parent.as_posix()
    df = pd.read_csv(csv_path)
    # Expect columns: ['filepath','label','source','y_true','prob_cancer','split']
    req = {"filepath","label","source","y_true","prob_cancer","split"}
    if not req.issubset(df.columns):
        raise ValueError(f"{csv_path} missing required columns {req - set(df.columns)}")

    # Split
    val = df[df["split"].astype(str).str.lower() == "val"].copy()
    tst = df[df["split"].astype(str).str.lower() == "test"].copy()
    if len(val) == 0 or len(tst) == 0:
        raise ValueError(f"{csv_path} must contain both VAL and TEST rows")

    yv = (val["y_true"].values).astype(int)
    yt = (tst["y_true"].values).astype(int)
    pv = val["prob_cancer"].astype(float).values
    pt = tst["prob_cancer"].astype(float).values

    # --- Platt on VAL ---
    a, b = platt_calibrate_fit(yv, pv)
    pv_cal = platt_apply(pv, a, b)
    pt_cal = platt_apply(pt, a, b)

    # --- pick τ* on calibrated VAL ---
    tau_star, policy = pick_tau_on_val(yv, pv_cal, prec_target=0.98)

    # --- metrics ---
    val_metrics  = metrics_at_threshold(yv, pv_cal, tau_star)
    test_metrics = metrics_at_threshold(yt, pt_cal, tau_star)

    print(f"\n[{Path(fold_dir).name}] NEW τ*={tau_star:.6f} policy={policy}")
    print("VAL :", val_metrics)
    print("TEST:", test_metrics)

    # pooled containers
    pooled_y.append(yt)
    pooled_p.append(pt_cal)
    pooled_taus.append(np.full_like(yt, fill_value=tau_star, dtype=float))

    # per-image TEST rows with τ used
    dft = tst[["filepath","source","label","y_true"]].copy()
    dft["prob_cancer_cal_platt"] = pt_cal
    dft["tau_used"] = tau_star
    dft["tau_policy"] = policy
    dft["fold_dir"] = fold_dir
    concat_rows.append(dft)

    per_fold_summary.append({
        "fold_dir": fold_dir,
        "tau_star": tau_star,
        "policy": policy,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics
    })

# ------------------------------------------------------------
# Pooled metrics (per-fold τ* applied per sample)
# ------------------------------------------------------------
Y = np.concatenate(pooled_y, axis=0)
P = np.concatenate(pooled_p, axis=0)
T = np.concatenate(pooled_taus, axis=0)

pooled = metrics_at_threshold(Y, P, 0.5)  # reference @0.5 (optional)
pooled_tau = metrics_at_threshold(Y, P, 0.0)  # dummy to init
# But we need per-sample τ*, so compute yhat with elementwise thresholds
yhat_tau = (P >= T).astype(int)
pr, rc, f1, _ = precision_recall_fscore_support(Y, yhat_tau, average="binary", zero_division=0)
try:
    auc = roc_auc_score(Y, P)
except Exception:
    auc = float("nan")
acc = (yhat_tau == Y).mean()
cm = confusion_matrix(Y, yhat_tau, labels=[0,1]).tolist()

pooled_tau = {"acc": float(acc), "prec": float(pr), "rec": float(rc), "f1": float(f1), "auc": float(auc), "cm": cm}

print("\n=== Pooled TEST with per-fold τ* (Platt-calibrated) ===")
print(pooled_tau)

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------
concat_df = pd.concat(concat_rows, ignore_index=True)
csv_out = os.path.join(OUT, "effnet_kagglecv_test_concat_tauCAL.csv")
concat_df.to_csv(csv_out, index=False)

with open(os.path.join(OUT, "effnet_kagglecv_metrics_tauCAL.json"), "w") as f:
    json.dump({
        "per_fold": per_fold_summary,
        "pooled_test_tauStar": pooled_tau
    }, f, indent=2)

print("\nSaved:")
print(" -", csv_out)
print(" -", os.path.join(OUT, "effnet_kagglecv_metrics_tauCAL.json"))


Found fold runs:
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_1/effnet_fold1
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_1/effnet_fold1_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_2/effnet_fold2_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_3/effnet_fold2_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_4/effnet_fold2_imagenet
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Kaggle_CV/fold_5/effnet_fold2_imagenet

[effnet_fold1] NEW τ*=0.599990 policy=prec>=0.98
VAL : {'acc': 0.6, 'prec': 1.0, 'rec': 0.3333333333333333, 'f1': 0.5, 'auc': 0.6666666666666666, 'cm': [[2, 0], [2, 1]]}
TEST: {'acc': 0.45454545454545453, 'prec': 0.5, 'rec': 0.5, 'f1': 0.5, 'auc': 0.366666666